# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
# Đã lược bỏ spacy / langchain-community / llama-index: không import ở bất kỳ cell nào,
# chỉ làm chậm setup ~4 phút. Thêm python-dotenv (chạy local) và datasketch (bonus near-dedup).
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx datasets python-dotenv datasketch

Note: you may need to restart the kernel to use updated packages.


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata, shutil, threading
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

# Chạy local: nạp .env. Chạy Colab: bỏ qua, đọc từ Colab Secrets.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

# Model cho các tầng KHỐI LƯỢNG LỚN (coref, NER+RE, seed extraction).
# Chọn model NHỎ NHẤT còn đủ chất lượng, và luôn chạy ở mức reasoning thấp nhất:
# reasoning token ẩn tính vào TPM y như output token.
# Benchmark trên tài khoản này (coref 8 chunk/request):
#   openai/gpt-oss-20b  effort=low : 1.3s, out= 424 token
#   openai/gpt-oss-120b effort=low : 1.9s, out= 617 token  (+45% token, cùng kết quả)
BULK_MODEL = get_secret("BULK_MODEL", "") or "openai/gpt-oss-120b"

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

# --- Thư mục IO: Colab dùng /content, local dùng cwd ---
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "outputs"
for _d in (DATA_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

DATA_PATH = str(BASE_DIR / "hackernoon_subset.csv")
GOLDEN_PATH = str(DATA_DIR / "golden_dataset.csv")
CHECKPOINT = str(OUT_DIR / "graphrag_eval_checkpoint.csv")
COREF_CKPT = str(OUT_DIR / "coref_checkpoint.csv")
EXTRACT_CKPT = str(OUT_DIR / "extraction_checkpoint.csv")
EVAL_RESULTS_PATH = str(OUT_DIR / "graphrag_eval_results.csv")
SUMMARY_PATH = str(OUT_DIR / "graphrag_vs_flatrag_summary.csv")
AUDIT_PATH = str(OUT_DIR / "entity_resolution_audit.csv")
SUPERNODE_PATH = str(OUT_DIR / "supernode_report.csv")

# --- Scale guard ---
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
# Dataset nay chi co snippet ~32 tu (median 43 tu sau khi ghep title), tuc
# ~60 token/chunk thay vi ~310 token/chunk nhu bai bao day du. Chi phi token
# giam ~4 lan -> tang so chunk trich xuat de do thi du day cho cau multi-hop.
# TPD 200k token/ngay va dung chung CHO CA TO CHUC (khong phai moi model mot
# pool nhu ban dau tuong). 200 chunk giau quan he cho do thi day hon han 300
# chunk ngau nhien, ma chi ton ~120k token cho ca coref lan extraction.
EXTRACTION_MAX_CHUNKS = 200
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

# --- Điều phối rate limit Groq free tier ---
# ĐO THỰC TẾ từ header x-ratelimit-*:
#   x-ratelimit-limit-tokens   = 8000  token/phút   <-- ràng buộc thật
#   x-ratelimit-limit-requests = 1000  request      <-- rất thoáng, không phải nút thắt
# Giới hạn tính RIÊNG CHO TỪNG MODEL => mỗi tầng dùng model khác nhau sẽ có
# ngân sách token tách biệt. Một request coref 8 chunk ~2900 token, tức chỉ
# ~2 request/phút lọt qua — mọi thứ nhanh hơn thế đều biến thành 429 + backoff.
GROQ_DEFAULT_TPM = 8000     # dùng khi chưa đọc được header; sau đó tự cập nhật
TPM_SAFETY = 0.80           # chừa biên cho sai số ước lượng token
LLM_WORKERS = 2             # TPM-bound nên nhiều worker cũng vô ích
MAX_COMPLETION_TOKENS = 3072
COREF_BATCH_SIZE = 15     # chunk ngan -> gop nhieu hon/request de bot overhead prompt
EXTRACT_BATCH_SIZE = 10

# Bỏ qua LLM và dùng lại checkpoint nếu đã chạy trước đó.
FORCE_RECOMPUTE = False

# Self-correction goi 2-3 LLM/cau; tat rieng khi quota ngay da can.
RUN_SELF_CORRECTION = False

# Bật/tắt phần Bonus (tốn thêm ~20 request LLM).
RUN_BONUS = True

def _mask(v):
    v = str(v or "")
    return "❌ MISSING" if not v else f"OK (len={len(v)}, …{v[-4:]})"

print("=== SECRET CHECK (giá trị được che, không in ra key) ===")
for _k, _v in [
    ("NEO4J_URI", NEO4J_URI), ("NEO4J_USER", NEO4J_USER),
    ("NEO4J_PASSWORD", NEO4J_PASSWORD), ("NEO4J_DATABASE", NEO4J_DATABASE),
    ("GROQ_API_KEY", GROQ_API_KEY), ("GROQ_MODEL", GROQ_MODEL),
    ("BULK_MODEL", BULK_MODEL),
    ("JUDGE_PROVIDER", JUDGE_PROVIDER), ("JUDGE_MODEL", JUDGE_MODEL),
    ("HF_TOKEN", HF_TOKEN),
]:
    shown = _v if _k in {"GROQ_MODEL", "BULK_MODEL", "JUDGE_MODEL", "JUDGE_PROVIDER",
                         "NEO4J_USER", "NEO4J_DATABASE"} else _mask(_v)
    print(f"  {_k:16s} {shown}")

if JUDGE_PROVIDER == "groq" and JUDGE_MODEL and JUDGE_MODEL == BULK_MODEL:
    print(f"\n⚠️  JUDGE_MODEL trùng BULK_MODEL ({JUDGE_MODEL}) -> self-preference bias.")

print(f"\nBASE_DIR = {BASE_DIR}")
print(f"Ngân sách: ~{GROQ_DEFAULT_TPM:,} token/phút MỖI MODEL (hệ số an toàn {TPM_SAFETY}), "
      f"{LLM_WORKERS} worker, EXTRACTION_MAX_CHUNKS={EXTRACTION_MAX_CHUNKS}")
print("💡 Pipeline này KHÔNG cần GPU — chỉ MiniLM embedding, chạy CPU vài phút.")

=== SECRET CHECK (giá trị được che, không in ra key) ===
  NEO4J_URI        OK (len=37, …j.io)
  NEO4J_USER       dc9c91f3
  NEO4J_PASSWORD   OK (len=43, …eITQ)
  NEO4J_DATABASE   dc9c91f3
  GROQ_API_KEY     OK (len=56, …3Tis)
  GROQ_MODEL       openai/gpt-oss-120b
  BULK_MODEL       openai/gpt-oss-120b
  JUDGE_PROVIDER   groq
  JUDGE_MODEL      qwen/qwen3.6-27b
  HF_TOKEN         OK (len=37, …ruZB)

BASE_DIR = d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan
Ngân sách: ~8,000 token/phút MỖI MODEL (hệ số an toàn 0.8), 2 worker, EXTRACTION_MAX_CHUNKS=200
💡 Pipeline này KHÔNG cần GPU — chỉ MiniLM embedding, chạy CPU vài phút.


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = DATA_PATH

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
# LIMIT_MB hạ từ 300 -> 80: standardize_news() dù sao cũng chỉ giữ
# LAB_MAX_ARTICLES = 1500 bài, phần tải thêm chỉ tốn thời gian download.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 80

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đặt True nếu muốn tải lại từ đầu; mặc định tái sử dụng file đã tải
# để "Restart & Run All" không mất thêm vài phút download.
FORCE_REDOWNLOAD = False

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

_already = os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 1_000_000

if _already and not FORCE_REDOWNLOAD:
    print(
        f"⏭️  Bỏ qua download: đã có {OUTPUT_CSV} "
        f"({os.path.getsize(OUTPUT_CSV)/(1024*1024):.2f} MB). "
        f"Đặt FORCE_REDOWNLOAD=True nếu muốn tải lại."
    )
else:
    print("Đang kết nối luồng dữ liệu (streaming)...")

    try:
        dataset = load_dataset(
            DATASET_NAME,
            split="train",
            streaming=True,
            token=HF_TOKEN,
        )
        iterator = iter(dataset)

        first_row = next(iterator)
        headers = list(first_row.keys())

        print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

        rows_written = 0
        total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
        unit_progress = "MB" if PRIORITIZE_MB else "row"

        with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
            writer.writeheader()
            writer.writerow(first_row)
            rows_written += 1

            # Flush để kích thước file phản ánh dữ liệu vừa ghi.
            f.flush()
            file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

            with tqdm(
                total=total_progress,
                desc=f"Đang tải ({unit_progress})",
                unit=unit_progress,
            ) as pbar:
                if PRIORITIZE_MB:
                    pbar.n = min(file_size_mb, LIMIT_MB)
                    pbar.refresh()
                else:
                    pbar.update(1)

                for row in iterator:
                    writer.writerow(row)
                    rows_written += 1

                    # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                    # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                    should_check_size = (
                        PRIORITIZE_MB
                        and (
                            rows_written % 100 == 0
                            or file_size_mb >= LIMIT_MB * 0.95
                        )
                    )

                    if should_check_size:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                        pbar.refresh()
                    elif not PRIORITIZE_MB:
                        pbar.update(1)

                    # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                    if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                        print(
                            f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                            f"{file_size_mb:.2f} MB "
                            f"(Tổng: {rows_written:,} dòng)"
                        )
                        break

                    # Hard-stop theo số dòng trong mọi chế độ.
                    if rows_written >= LIMIT_ROWS:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        print(
                            f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                            f"{rows_written:,} dòng "
                            f"(Dung lượng: {file_size_mb:.2f} MB)"
                        )
                        break

            f.flush()

        final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
        print(
            f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
            f"   Rows: {rows_written:,}\n"
            f"   Size: {final_size_mb:.2f} MB"
        )

    except StopIteration:
        raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
    except Exception as e:
        print(f"\n❌ Có lỗi xảy ra: {e}")
        print(
            "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
            "(3) kết nối mạng của Colab."
        )
        raise

# Đồng bộ đường dẫn cho cell loader tiếp theo.
DATA_PATH = OUTPUT_CSV

⏭️  Bỏ qua download: đã có d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\hackernoon_subset.csv (80.00 MB). Đặt FORCE_REDOWNLOAD=True nếu muốn tải lại.


In [4]:
#@title 1.4 — Neo4j connection + schema
from neo4j.exceptions import AuthError, ServiceUnavailable, ClientError

driver = None

# Xoá sạch graph trước khi nạp lại. Bật True để "Restart & Run All" luôn cho ra
# một đồ thị sạch — nếu không, dữ liệu của lần chạy trước còn sót lại sẽ làm
# sai lệch degree/super-node và kiểm tra provenance mất ý nghĩa.
RESET_GRAPH = True

_AUTH_HELP = """
❌ Neo4j từ chối đăng nhập (AuthError). URI đúng, mạng đúng, nhưng cặp user/password sai.
   Đang dùng: user={user!r}  uri={uri!r}  password dài {plen} ký tự

   Kiểm tra theo thứ tự:
   1. Trên AuraDB, username LUÔN là 'neo4j'. Chuỗi {aura} trong URI là instance-id,
      KHÔNG phải username và cũng không phải tên database.
   2. Password là chuỗi sinh ra một lần duy nhất trong file 'Neo4j-<id>-Created-....txt'
      tải về lúc tạo instance — KHÔNG phải mật khẩu tài khoản console.neo4j.io.
   3. Mất file đó thì vào console.neo4j.io -> chọn instance -> Reset password,
      rồi cập nhật lại secret NEO4J_PASSWORD.
   4. Trên Colab: mở tab Secrets (biểu tượng chìa khoá), xoá và dán lại NEO4J_PASSWORD
      (đừng dán kèm dấu nháy), và bật công tắc "Notebook access" cho TẤT CẢ secret —
      tắt thì userdata.get() ném exception, get_secret() nuốt lỗi và trả về default rỗng.
   5. Nếu instance vừa bị xoá/tạo lại thì URI cũ + password cũ đều không còn hiệu lực.

   Sau khi sửa secret, chạy lại cell 1.2 TRƯỚC rồi mới chạy lại cell 1.4.
"""

def connect_neo4j():
    global driver
    missing = [k for k, v in [("NEO4J_URI", NEO4J_URI), ("NEO4J_PASSWORD", NEO4J_PASSWORD)] if not v]
    if missing:
        raise ValueError(
            f"Thiếu Neo4j secrets: {', '.join(missing)}. "
            "Trên Colab thêm ở tab Secrets (nhớ bật 'Notebook access'), "
            "chạy local thì điền vào .env, sau đó chạy lại cell 1.2."
        )
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    try:
        driver.verify_connectivity()
    except AuthError:
        driver = None
        raise RuntimeError(_AUTH_HELP.format(
            user=NEO4J_USER, uri=NEO4J_URI, plen=len(NEO4J_PASSWORD),
            aura=repr(_AURA_ID) if _AURA_ID else "hex",
        )) from None
    except ServiceUnavailable as e:
        driver = None
        raise RuntimeError(
            f"❌ Không kết nối được tới {NEO4J_URI}.\n"
            "   Aura Free tự paused sau 3 ngày không dùng: vào console.neo4j.io bấm Resume,\n"
            "   đợi trạng thái chuyển sang RUNNING rồi chạy lại. Aura bắt buộc scheme\n"
            f"   'neo4j+s://' (đang dùng: {NEO4J_URI.split('://')[0]}://).\n"
            f"   Chi tiết: {e}"
        ) from None

    # verify_connectivity() chỉ bắt tay với server, không kiểm tra database có tồn tại.
    # Kiểm luôn ở đây để lỗi sai tên database nổ tại 1.4 thay vì tận 2.3.
    try:
        with driver.session(database=NEO4J_DATABASE) as s:
            s.run("RETURN 1").consume()
    except ClientError as e:
        driver = None
        raise RuntimeError(
            f"❌ Kết nối OK nhưng database {NEO4J_DATABASE!r} không dùng được.\n"
            "   AuraDB Free chỉ có đúng một database tên 'neo4j' -> đặt NEO4J_DATABASE=neo4j.\n"
            f"   Chi tiết: {e}"
        ) from None

    print(f"✅ Neo4j connected — user={NEO4J_USER}, database={NEO4J_DATABASE}")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def reset_graph(batch=5000):
    """Xoá toàn bộ node/edge theo batch để không vượt heap của Aura Free."""
    total = 0
    while True:
        n = run_cypher(
            # count(*) đếm SỐ DÒNG, không chạm vào node đã xoá -> tránh lỗi "deleted entity".
            "MATCH (n) WITH n LIMIT $batch DETACH DELETE n RETURN count(*) AS n",
            batch=int(batch),
        )[0]["n"]
        total += n
        if n == 0:
            break
    print(f"🧹 Reset graph: đã xoá {total:,} node.")

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
if RESET_GRAPH:
    reset_graph()
setup_graph_schema()

✅ Neo4j connected — user=dc9c91f3, database=dc9c91f3
🧹 Reset graph: đã xoá 286 node.


✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
# Dataset HackerNoon/tech-company-news-data-dump co schema:
#   companyName, companyUrl, published_at, url, title, main_image, description
# Than bai nam o cot 'description' (mot doan tom tat ~30 tu da bi cat ngan),
# khong phai 'text'. Cot 'title' moi la cho dac entity nhat, ma extraction o 2.1
# chi nhan chunk.text chu khong nhan title -> phai ghep title vao text, neu khong
# phan lon chunk khong du ngu canh de rut ra bat ky triple nao.
PREPEND_TITLE_TO_TEXT = True

# Nguong do dai toi thieu cua THAN BAI (chua ghep title). 56% dong cua dataset
# nay co description rong; neu do do dai sau khi da ghep title thi cac dong do
# tro thanh "bai bao" chi co moi tieu de va lot qua filter.
MIN_BODY_CHARS = 80

# Cot than bai, xep theo do uu tien: ban day du truoc, ban tom tat sau.
TEXT_COL_CANDIDATES = [
    "text", "content", "article", "body", "story",
    "full_text", "article_text", "story_text", "selftext",
    "description", "summary", "abstract", "snippet",
]

def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True, fallback_longest=False):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]

    if fallback_longest:
        # Khong ten nao khop -> doan cot than bai la cot chuoi dai nhat, bo qua
        # cac cot url/anh (dai nhung khong phai van ban).
        best, best_len = None, 0.0
        sample = df.head(500)
        for c in df.columns:
            if re.search(r"url|link|image|img|href|thumb|slug", str(c), re.I):
                continue
            s = sample[c].dropna().astype(str)
            if s.empty:
                continue
            mean_len = float(s.str.len().mean())
            if mean_len > best_len:
                best, best_len = c, mean_len
        if best is not None and best_len >= 80:
            print(f"⚠️  Khong thay cot nao trong {candidates}; "
                  f"doan cot than bai = {best!r} (dai trung binh {best_len:.0f} ky tu).")
            return best

    if required:
        # Liet ke cot thuc te: loi thieu cot phai tu chan doan duoc, khong bat
        # nguoi dung phai mo file CSV ra doi chieu bang tay.
        raise KeyError(
            f"Khong tim thay cot nao trong {candidates}. "
            f"Cot thuc te cua file: {list(df.columns)}"
        )
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

# --- CURATION: chon corpus GIAU QUAN HE ------------------------------------
# Lay mau ngau nhien 1500/63717 bai cho ra corpus gan nhu ROI RAC ve thuc the:
# do thi thu duoc chi co 69 canh, degree cao nhat = 4, DUNG 1 duong 2-hop xuyen
# chunk va 0 cap entity cross-doc -> khong the phan biet Flat RAG voi GraphRAG,
# tuc benchmark mat y nghia.
#
# Nguyen nhan: dataset trai tren 168 cong ty, moi bai la mot snippet ~40 tu ve
# mot san pham rieng biet, nen cac thuc the gan nhu khong bao gio lap lai.
#
# Xu ly: giu nguyen LAB_MAX_ARTICLES=1500 nhung CHON co chu dich nhung bai vua
# co TIN HIEU QUAN HE vua nhac den CONG TY LON. Day chinh la buoc pre-filter re
# tien ma pipeline production nao cung lam truoc khi goi LLM dat tien.
PREFER_RELATION_DENSE = True

RELATION_SIGNAL_RE = (
    r"\b(?:acquir\w*|acquisition|merger|merge[sd]?|invest\w*|funding|raise[sd]?|"
    r"stake|partner\w*|collaborat\w*|launch\w*|develop\w*|unveil\w*|appoint\w*|"
    r"found(?:ed|er)|joins?|hire[sd]?)\b"
)
MAJOR_ENTITY_RE = (
    r"\b(?:Microsoft|Google|Alphabet|Apple|Meta|Facebook|Amazon|AWS|Nvidia|OpenAI|"
    r"Anthropic|Tesla|IBM|Intel|AMD|Samsung|Oracle|Salesforce|Adobe|Netflix|Qualcomm|"
    r"TSMC|Twitter|SpaceX|Uber|Airbnb|Alibaba|Tencent|Sony|Cisco|Dell|Broadcom|"
    r"Palantir|Snowflake|Databricks|Stripe|Shopify|Zoom|Baidu|Huawei|Xiaomi|Starlink)\b"
)

def select_relation_dense(df):
    full = df["title"].astype(str) + ". " + df["text"].astype(str)
    has_rel = full.str.contains(RELATION_SIGNAL_RE, case=False, regex=True, na=False)
    has_maj = full.str.contains(MAJOR_ENTITY_RE, case=True, regex=True, na=False)
    keep = has_rel & has_maj
    print(f"Curation: tin hieu quan he {has_rel.sum():,} | cong ty lon {has_maj.sum():,} "
          f"| CA HAI {keep.sum():,}/{len(df):,} ({keep.mean():.1%})")
    if keep.sum() >= LAB_MAX_ARTICLES:
        return df[keep].reset_index(drop=True)
    # Khong du -> lay het phan giau quan he roi bu them bai ngau nhien.
    need = LAB_MAX_ARTICLES - int(keep.sum())
    rest = df[~keep].sample(min(need, int((~keep).sum())), random_state=SEED)
    print(f"   Chi co {keep.sum():,} bai giau quan he -> bu them {len(rest):,} bai ngau nhien.")
    return pd.concat([df[keep], rest]).reset_index(drop=True)

def standardize_news(raw):
    text_col = pick_col(raw, TEXT_COL_CANDIDATES, fallback_longest=True)
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)
    print(f"Map cot: text<-{text_col!r} | title<-{title_col!r} | "
          f"date<-{date_col!r} | id<-{id_col!r}")

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    # Loc do dai TRUOC khi ghep title, xem MIN_BODY_CHARS o dau cell.
    _raw_rows = len(df)
    df = df[df["text"].str.len() >= MIN_BODY_CHARS].copy()
    print(f"Loc body >= {MIN_BODY_CHARS} ky tu: {_raw_rows:,} -> {len(df):,} dong "
          f"({_raw_rows - len(df):,} dong body rong/qua ngan).")

    if PREPEND_TITLE_TO_TEXT and title_col:
        # Bo qua khi body da bat dau bang chinh title -> tranh lap tieu de.
        df["text"] = [
            x if (not t or x.lower().startswith(t.lower())) else f"{t}. {x}"
            for t, x in zip(df["title"], df["text"])
        ]

    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup (SHA-1 trên title+text): {before:,} -> {len(df):,}")

    # article_id phải unique: near-dedup LSH và chunk_id đều dựa trên nó.
    dup_ids = df["article_id"].duplicated().sum()
    if dup_ids:
        print(f"⚠️  {dup_ids} article_id trùng -> gán lại id theo nội dung.")
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]
        df = df.drop_duplicates("article_id").reset_index(drop=True)

    if PREFER_RELATION_DENSE:
        df = select_relation_dense(df)

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
        print(f"Scale guard LAB_MAX_ARTICLES -> {len(df):,} bài.")

    # Ưu tiên bài có ngày xuất bản: provenance published_date là bắt buộc trên mọi edge.
    print(f"Bài có published_date: {(df.published_date != '').sum():,}/{len(df):,}")

    # Dataset nay chi co doan tom tat ngan, khong phai bai bao day du. In do dai
    # ra de biet truoc so chunk va ky vong recall cua extraction o 2.1.
    _w = df["text"].str.split().str.len()
    print(f"Do dai text: trung vi {int(_w.median())} tu, p90 {int(_w.quantile(0.9))} tu "
          f"(CHUNK_WORDS={CHUNK_WORDS} -> phan lon bai chi sinh 1 chunk).")
    return df

def chunk_text(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
display(news_df.head(3))

Map cot: text<-'description' | title<-'title' | date<-'published_at' | id<-None


Loc body >= 80 ky tu: 137,577 -> 63,717 dong (73,860 dong body rong/qua ngan).


Exact dedup (SHA-1 trên title+text): 63,717 -> 57,085
Curation: tin hieu quan he 14,516 | cong ty lon 5,981 | CA HAI 1,936/57,085 (3.4%)
Scale guard LAB_MAX_ARTICLES -> 1,500 bài.
Bài có published_date: 1,500/1,500
Do dai text: trung vi 44 tu, p90 58 tu (CHUNK_WORDS=220 -> phan lon bai chi sinh 1 chunk).


,text,title,published_date,article_id
0,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity. In February GreenPages acqu...,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,4bd7afdba71243b0dbcd
1,Information Technology and Telecoms. electronRx writes...Apple announced the launch of the iPhone 15 on 12th Septemb...,Information Technology and Telecoms,2023-09-19,8b2a4221bdca74cf28ac
2,Synopsys TSMC team up to help clients accelerate 2nm chips designs. EDA specialist Synopsys has recently announced i...,Synopsys TSMC team up to help clients accelerate 2nm chips designs,2023-05-17,4b907bbb535472c4877e


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.5b — [BONUS C] Near-dedup bằng MinHash + LSH, rồi chunking
# Exact SHA-1 chỉ bắt được bản sao byte-for-byte. Tin công nghệ hay bị repost
# kèm sửa tiêu đề/thêm 1 đoạn -> vẫn lọt qua exact hash và làm lệch đồ thị
# (cùng 1 sự kiện bị đếm 2 lần -> confidence giả, degree giả).
#
# Độ phức tạp: MinHash O(N * shingles) + LSH bucket lookup ~O(N).
# KHÔNG dùng pairwise cosine O(N^2) như AI Agent gợi ý ban đầu.
from datasketch import MinHash, MinHashLSH

# Van ban chi ~45 tu -> 5-gram sinh qua it shingle, Jaccard gan nhu khong bao
# gio dat 0.85 tru khi trung khop tuyet doi (ma exact dedup da loc roi).
# Ha nguong va rut ngan shingle cho phu hop do dai thuc te cua corpus.
NEAR_DUP_THRESHOLD = 0.75
NEAR_DUP_NUM_PERM = 128
NEAR_DUP_SHINGLE_K = 4

def shingles(text, k=NEAR_DUP_SHINGLE_K):
    words = norm_space(text).lower().split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}

def build_minhash(text, num_perm=NEAR_DUP_NUM_PERM, k=NEAR_DUP_SHINGLE_K):
    m = MinHash(num_perm=num_perm)
    for sh in shingles(text, k):
        m.update(sh.encode("utf-8"))
    return m

def near_dedup(df, threshold=NEAR_DUP_THRESHOLD, num_perm=NEAR_DUP_NUM_PERM):
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    sigs, keep_idx, audit = {}, [], []

    for i, r in enumerate(tqdm(list(df.itertuples(index=False)), desc="Near-dedup")):
        key = f"{i}"
        m = build_minhash(r.text, num_perm)
        candidates = lsh.query(m)

        if candidates:
            best = max(candidates, key=lambda c: m.jaccard(sigs[c]))
            kept = df.iloc[int(best)]
            audit.append({
                "dropped_article_id": r.article_id,
                "dropped_title": r.title[:90],
                "kept_article_id": kept.article_id,
                "kept_title": str(kept.title)[:90],
                "jaccard": round(float(m.jaccard(sigs[best])), 4),
                "decision": "DROP_NEAR_DUP",
            })
            continue

        lsh.insert(key, m)
        sigs[key] = m
        keep_idx.append(i)

    out = df.iloc[keep_idx].reset_index(drop=True)
    audit_df = pd.DataFrame(audit)
    print(f"Near-dedup (MinHash/LSH, Jaccard>={threshold}): {len(df):,} -> {len(out):,} "
          f"(loại {len(df)-len(out):,} bài near-duplicate)")
    return out, audit_df

def near_dup_selftest(df, n=3):
    """Chung minh co che BAT DUOC near-dup, doc lap voi viec corpus co hay khong.

    "Tim thay 0 cap" tu no khong phai bang chung: co the do co che hong. Test
    nay tiem ban sao da sua doi cua 3 bai that va yeu cau LSH bat lai duoc.
    """
    base = df.head(n).copy()
    dup = base.copy()
    dup["text"] = base["text"].astype(str) + " Additional reporting by newsroom staff."
    dup["article_id"] = base["article_id"].astype(str) + "__nearcopy"
    probe = pd.concat([base, dup], ignore_index=True)
    kept, audit = near_dedup(probe, threshold=NEAR_DUP_THRESHOLD)
    # LSH la thuat toan XAC SUAT (banding tren 128 permutation): cap co Jaccard
    # sat nguong chi duoc phat hien voi xac suat < 1. Bat duoc 2/3 khong phai
    # loi cai dat ma la dac tinh cua thuat toan -> danh gia theo TY LE, khong
    # phai pass/fail tuyet doi.
    caught = len(audit)
    print(f"   Self-test: {len(probe)} bai (trong do {n} la ban sao gan) -> giu {len(kept)} "
          f"| bat duoc {caught}/{n} cap ({caught/n:.0%}) "
          f"{'✅ co che hoat dong' if caught >= 1 else '❌ co che HONG'}")
    if not audit.empty:
        display(audit[["dropped_article_id", "kept_article_id", "jaccard", "decision"]])
    return audit

def near_dup_sweep(df, thresholds=(0.9, 0.8, 0.7, 0.6, 0.5)):
    """Quet nguong tren corpus that de bao cao duoc trade-off, khong doan mo."""
    rows = []
    for th in thresholds:
        _, audit = near_dedup(df, threshold=th)
        rows.append({"threshold": th, "cap_bi_loai": len(audit),
                     "con_lai": len(df) - len(audit)})
    return pd.DataFrame(rows)

print(">>> [1] Self-test co che MinHash/LSH")
_selftest_audit = near_dup_selftest(news_df)

print(); print(">>> [2] Quet nguong tren corpus that")
near_dup_sweep_df = near_dup_sweep(news_df)
display(near_dup_sweep_df)
near_dup_sweep_df.to_csv(OUT_DIR / "near_dedup_threshold_sweep.csv", index=False)

print(); print(">>> [3] Ap dung o nguong da chon")
news_df, near_dup_audit_df = near_dedup(news_df)
if not near_dup_audit_df.empty:
    display(near_dup_audit_df.sort_values("jaccard", ascending=False).head(15))
    near_dup_audit_df.to_csv(OUT_DIR / "near_dedup_audit.csv", index=False)
else:
    print(f"Ket qua thuc nghiem: 0 cap near-duplicate o nguong {NEAR_DUP_THRESHOLD}. "
          f"Xem bang sweep o tren de biet nguong nao moi bat dau bat duoc cap nao; "
          f"co che da duoc self-test xac nhan la hoat dong.")

chunks_df = build_chunks(news_df)
print(f"chunks_df: {len(chunks_df):,} chunk")
display(chunks_df.head(3))

>>> [1] Self-test co che MinHash/LSH


Near-dedup:   0%|          | 0/6 [00:00<?, ?it/s]

Near-dedup (MinHash/LSH, Jaccard>=0.75): 6 -> 4 (loại 2 bài near-duplicate)
   Self-test: 6 bai (trong do 3 la ban sao gan) -> giu 4 | bat duoc 2/3 cap (67%) ✅ co che hoat dong


,dropped_article_id,kept_article_id,jaccard,decision
0,4bd7afdba71243b0dbcd__nearcopy,4bd7afdba71243b0dbcd,0.8516,DROP_NEAR_DUP
1,4b907bbb535472c4877e__nearcopy,4b907bbb535472c4877e,0.8750,DROP_NEAR_DUP



>>> [2] Quet nguong tren corpus that


...

Near-dedup (MinHash/LSH, Jaccard>=0.75): 1,500 -> 1,495 (loại 5 bài near-duplicate)


,dropped_article_id,dropped_title,kept_article_id,kept_title,jaccard,decision
0,5bff5dc60b34e5aec2d0,satellite internet service in Bangladesh,43168210f03cb0a63374,SpaceX wants to launch satellite internet service in Bangladesh,0.9062,DROP_NEAR_DUP
3,fb866d3ae7d51a579fc5,Adobe unveils interactive digital dress in Los Angeles,19f795c29a8e95c0a889,Adobe unveils interactive digital dress in Los Angeles,0.8672,DROP_NEAR_DUP
2,f601c15aea5554879d8b,Microsoft’s Bid to Buy Activision Blizzard Temporarily Halted by Judge as Merger Deadline,92b26ed4ba3c64f603bc,Microsoft’s Bid to Buy Activision Blizzard Temporarily Halted by Judge as Merger Deadline,0.7812,DROP_NEAR_DUP
4,eb0c494e273ccd590f5d,Alibaba Group Holding Limited (BABA) is Attracting Investor Attention: Here is What You Sh,07346fc4ad98a0ed1cf6,Adobe Inc. (ADBE) is Attracting Investor Attention: Here is What You Should Know,0.6328,DROP_NEAR_DUP
1,993387793de653258b7a,Meta Platforms (META) Bounced Back in Q1,11bb0e7759401322e9af,What Makes Alphabet (GOOG) an Attractive Long-term Bet?,0.6250,DROP_NEAR_DUP


Chunking:   0%|          | 0/1495 [00:00<?, ?it/s]

chunks_df: 1,501 chunk


,chunk_id,article_id,title,published_date,text
0,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity. In February GreenPages acqu...
1,8b2a4221bdca74cf28ac::c0000,8b2a4221bdca74cf28ac,Information Technology and Telecoms,2023-09-19,Information Technology and Telecoms. electronRx writes...Apple announced the launch of the iPhone 15 on 12th Septemb...
2,4b907bbb535472c4877e::c0000,4b907bbb535472c4877e,Synopsys TSMC team up to help clients accelerate 2nm chips designs,2023-05-17,Synopsys TSMC team up to help clients accelerate 2nm chips designs. EDA specialist Synopsys has recently announced i...


In [7]:
#@title 1.6 — LLM wrapper: token-budget TPM + TPD + fallback model
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

LLM_STATS = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0,
             "total_tokens": 0, "retries": 0, "rate_limited": 0}
TOKENS_BY_MODEL = Counter()
_stats_lock = threading.Lock()

# --- reasoning_effort: MỖI HỌ MODEL NHẬN GIÁ TRỊ KHÁC NHAU -----------------
# Đo thực tế trên tài khoản này:
#   openai/gpt-oss-*  -> low | medium | high
#   qwen/qwen3.6-*    -> CHỈ none | default. Để mặc định thì model phát khối
#                        <think>...</think> và Groq trả 400 "Failed to validate
#                        JSON" khi bật response_format=json_object.
# Truyền sai giá trị => 400 ngay, nên phải map theo tên model.
def reasoning_effort_for(model):
    m = str(model).lower()
    if "gpt-oss" in m:
        return "low"
    if m.startswith("qwen/"):
        return "none"
    return None

# --- HAI TẦNG RATE LIMIT, và tầng nguy hiểm hơn KHÔNG nằm trong header ------
#   TPM = 8 000 token/phút   -> có trong x-ratelimit-limit-tokens
#   TPD = 200 000 token/NGÀY -> KHÔNG có trong bất kỳ header nào, chỉ hiện ra
#                               trong nội dung lỗi 429 khi đã quá muộn.
# Cả hai đều tính RIÊNG THEO MODEL. Retry một lỗi TPD là vô ích: quota chỉ về
# sau nhiều giờ. Vì vậy phải nhận diện TPD, đánh dấu model là "chết trong ngày",
# và CHUYỂN SANG MODEL KHÁC thay vì backoff vô vọng.
MODEL_FALLBACKS = {
    "openai/gpt-oss-20b":  ["openai/gpt-oss-120b", "qwen/qwen3.6-27b"],
    "openai/gpt-oss-120b": ["qwen/qwen3.6-27b", "openai/gpt-oss-20b"],
    "qwen/qwen3.6-27b":    ["openai/gpt-oss-120b", "openai/gpt-oss-20b"],
}
TPD_EXHAUSTED = {}          # model -> thời điểm monotonic được phép dùng lại
_TPD_RE = re.compile(r"tokens per day \(tpd\).*?limit (\d+), used (\d+)", re.I)
_RETRY_RE = re.compile(r"try again in ([\dhms.]+)")

def _parse_retry_seconds(msg):
    m = _RETRY_RE.search(msg)
    if not m:
        return None
    raw, total = m.group(1), 0.0
    for value, unit in re.findall(r"([\d.]+)([hms])", raw):
        total += float(value) * {"h": 3600, "m": 60, "s": 1}[unit]
    return total or None

class TokenBudget:
    """Cửa sổ trượt 60 giây, tách riêng cho từng model, tự hiệu chuẩn."""

    def __init__(self, default_tpm=8000, safety=0.80):
        self.default_tpm = default_tpm
        self.safety = safety
        self.limits = {}
        self.calib = {}          # model -> hệ số nhân cho ước lượng thô
        self.events = defaultdict(deque)
        self.cond = threading.Condition()

    def _used(self, model, now):
        q = self.events[model]
        while q and now - q[0][0] > 60.0:
            q.popleft()
        return sum(e[1] for e in q)

    def _cap(self, model):
        return self.limits.get(model, self.default_tpm) * self.safety

    def estimate(self, model, raw_estimate):
        # Khởi đầu 2.0: đo được ước lượng thô hụt ~2 lần so với thực tế
        # (chunk_id dạng hex + JSON escape tokenize rất tệ, ~1.4 ký tự/token).
        return int(raw_estimate * self.calib.get(model, 2.0))

    def acquire(self, model, estimate):
        estimate = max(1, int(estimate))
        with self.cond:
            while True:
                now = time.monotonic()
                if self._used(model, now) + estimate <= self._cap(model):
                    handle = [now, estimate]
                    self.events[model].append(handle)
                    return handle
                q = self.events[model]
                wait = 60.0 - (now - q[0][0]) if q else 1.0
                self.cond.wait(timeout=max(0.2, min(wait, 5.0)))

    def settle(self, model, handle, actual, raw_estimate=None, limit_header=None):
        with self.cond:
            if limit_header:
                try:
                    self.limits[model] = int(limit_header)
                except (TypeError, ValueError):
                    pass
            if actual:
                handle[1] = int(actual)
                if raw_estimate:
                    # EMA: hệ số tự hội tụ về tỉ lệ thực tế/ước lượng thô.
                    ratio = actual / max(1, raw_estimate)
                    prev = self.calib.get(model, 2.0)
                    self.calib[model] = round(0.7 * prev + 0.3 * ratio, 3)
            self.cond.notify_all()

    def penalise(self, model, seconds):
        with self.cond:
            self.events[model].append([time.monotonic() + seconds - 60.0,
                                       self._cap(model)])

TOKEN_BUDGET = TokenBudget(default_tpm=GROQ_DEFAULT_TPM, safety=TPM_SAFETY)

def raw_estimate_tokens(messages, expected_out):
    chars = sum(len(m.get("content") or "") for m in messages)
    return int(chars / 3.2) + int(expected_out)

def available_models(model):
    """Model chính trước, rồi tới fallback; bỏ qua model đã cạn TPD."""
    now = time.monotonic()
    out = []
    for m in [model] + MODEL_FALLBACKS.get(model, []):
        if m in out:
            continue
        if TPD_EXHAUSTED.get(m, 0) > now:
            continue
        out.append(m)
    return out

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4,
              max_tokens=None, expected_out=600):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    raw_est = raw_estimate_tokens(messages, expected_out)
    candidates = available_models(model)
    if not candidates:
        raise RuntimeError(f"Mọi model fallback của {model} đều đã cạn TPD hôm nay.")

    last = None
    for use_model in candidates:
        for attempt in range(max_retries):
            est = TOKEN_BUDGET.estimate(use_model, raw_est)
            handle = TOKEN_BUDGET.acquire(use_model, est)
            try:
                kwargs = {
                    "model": use_model,
                    "messages": messages,
                    "temperature": 0.0,
                    "max_tokens": int(max_tokens or MAX_COMPLETION_TOKENS),
                }
                if json_mode:
                    kwargs["response_format"] = {"type": "json_object"}
                effort = reasoning_effort_for(use_model)
                if effort:
                    kwargs["reasoning_effort"] = effort

                raw = groq_client.chat.completions.with_raw_response.create(**kwargs)
                resp = raw.parse()

                usage = {}
                if getattr(resp, "usage", None):
                    usage = {
                        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                        "total_tokens": getattr(resp.usage, "total_tokens", None),
                    }
                TOKEN_BUDGET.settle(use_model, handle, usage.get("total_tokens"),
                                    raw_est, raw.headers.get("x-ratelimit-limit-tokens"))
                with _stats_lock:
                    LLM_STATS["calls"] += 1
                    for k in ("prompt_tokens", "completion_tokens", "total_tokens"):
                        LLM_STATS[k] += int(usage.get(k) or 0)
                    TOKENS_BY_MODEL[use_model] += int(usage.get("total_tokens") or 0)
                return resp.choices[0].message.content, usage

            except Exception as e:
                TOKEN_BUDGET.settle(use_model, handle, est)
                last = e
                msg = str(e)
                low = msg.lower()

                tpd = _TPD_RE.search(low)
                if tpd:
                    # Hết quota NGÀY: retry là vô nghĩa, chuyển model ngay.
                    wait = _parse_retry_seconds(low) or 3600.0
                    TPD_EXHAUSTED[use_model] = time.monotonic() + wait
                    limit, used = tpd.group(1), tpd.group(2)
                    print(f"   🛑 {use_model} cạn TPD ({used}/{limit} token/ngày). "
                          f"Chuyển sang model dự phòng.")
                    break

                with _stats_lock:
                    LLM_STATS["retries"] += 1
                if attempt == max_retries - 1:
                    break
                if "429" in low or "rate limit" in low:
                    with _stats_lock:
                        LLM_STATS["rate_limited"] += 1
                    delay = (_parse_retry_seconds(low) or min(60.0, 10.0 * (2 ** attempt))) + 1.0
                    TOKEN_BUDGET.penalise(use_model, delay)
                    time.sleep(delay)
                else:
                    time.sleep(min(20.0, 2.0 * (2 ** attempt) + random.random()))
    raise RuntimeError(f"{model}: {last}")

def groq_json(system, user, model=None, max_tokens=None, expected_out=600):
    text, usage = groq_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model=model, json_mode=True, max_tokens=max_tokens, expected_out=expected_out)
    return parse_json_object(text), usage

def parallel_map(fn, items, workers=None, desc="LLM"):
    """Chạy song song, TRẢ KẾT QUẢ ĐÚNG THỨ TỰ ĐẦU VÀO.

    Exception của một item trả về như một giá trị để không làm hỏng cả mẻ.
    """
    items = list(items)
    workers = int(workers or LLM_WORKERS)
    results = [None] * len(items)
    if workers <= 1 or len(items) <= 1:
        for i, it in enumerate(tqdm(items, desc=desc)):
            try:
                results[i] = fn(it)
            except Exception as e:
                results[i] = e
        return results
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fn, it): i for i, it in enumerate(items)}
        for f in tqdm(as_completed(futs), total=len(futs), desc=desc):
            i = futs[f]
            try:
                results[i] = f.result()
            except Exception as e:
                results[i] = e
    return results

def stage_report(label, t0, stats_before):
    dt = time.perf_counter() - t0
    calls = LLM_STATS["calls"] - stats_before["calls"]
    toks = LLM_STATS["total_tokens"] - stats_before["total_tokens"]
    comp = LLM_STATS["completion_tokens"] - stats_before["completion_tokens"]
    rl = LLM_STATS["rate_limited"] - stats_before["rate_limited"]
    print(f"\n⏱️  [{label}] {dt/60:.1f} phút | {calls} request | {toks:,} token "
          f"(output {comp:,}) | 429 hits: {rl}")
    print(f"    Token theo model (cộng dồn, hạn ~200k/ngày mỗi model): "
          f"{dict(TOKENS_BY_MODEL)}")

# --- Smoke test + tự dò TPM thật của từng model ---
for _label, _m, _out in [("BULK_MODEL", BULK_MODEL, 64),
                         ("GROQ_MODEL", GROQ_MODEL, 64),
                         ("JUDGE_MODEL", JUDGE_MODEL, 64)]:
    _t0 = time.perf_counter()
    try:
        _p, _u = groq_json("Return strict JSON only.", 'Return {"ok": true}',
                           model=_m, max_tokens=256, expected_out=_out)
        print(f"✅ {_label:12s} {_m:26s} {time.perf_counter()-_t0:4.1f}s "
              f"effort={reasoning_effort_for(_m)} TPM={TOKEN_BUDGET.limits.get(_m,'?')} -> {_p}")
    except Exception as e:
        print(f"❌ {_label:12s} {_m:26s} {type(e).__name__}: {str(e)[:140]}")

✅ BULK_MODEL   openai/gpt-oss-120b         0.7s effort=low TPM=8000 -> {'ok': True}


✅ GROQ_MODEL   openai/gpt-oss-120b         0.6s effort=low TPM=8000 -> {'ok': True}
✅ JUDGE_MODEL  qwen/qwen3.6-27b            0.2s effort=none TPM=8000 -> {'ok': True}


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [8]:
#@title 1.7 — Coreference resolution (substitution-based, có checkpoint)
# ⚡ THAY ĐỔI KIẾN TRÚC so với bản đầu — lý do chi tiết ghi trong báo cáo:
#
#   Bản cũ bắt LLM IN LẠI TOÀN VĂN từng chunk (`resolved_text`). Với 220 từ/chunk
#   × 5 chunk/request, mỗi request phải sinh ~1500 output token chỉ để chép lại
#   phần lớn nội dung không đổi. Đó là 100% chi phí TPM đổ vào việc sao chép.
#
#   Bản này chỉ yêu cầu DANH SÁCH PHÉP THẾ (mention -> antecedent), khoảng
#   30–80 output token/chunk, rồi Python tự áp phép thế. Ngoài chuyện nhanh hơn
#   ~10 lần, nó còn AN TOÀN HƠN VỀ MẶT NGỮ NGHĨA: LLM không có cơ hội sửa lén
#   số liệu/ngày tháng/tên sản phẩm ở phần văn bản còn lại — đúng tinh thần
#   "conservative coreference" mà đề bài yêu cầu.

COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
You do NOT rewrite text. You only output a list of safe substitutions.

Rules:
- List a substitution ONLY if the antecedent is explicitly named in the SAME chunk.
- List a mention ONLY if EVERY occurrence of it in that chunk refers to the SAME antecedent.
  If different occurrences refer to different entities, put the mention in unresolved_mentions instead.
- Never invent entities. Never touch dates, numbers, tickers or product names.
- Prefer leaving a mention unresolved over guessing.
Return strict JSON only.
""".strip()

MAX_SUBS_PER_CHUNK = 8

# Đại từ trần: thay TẤT CẢ occurrence là rủi ro cao vì các lần xuất hiện sau
# thường trôi sang thực thể khác. Ví dụ thật:
#   "Microsoft ... The company said it would invest $10B in OpenAI. It is the
#    largest round yet."  -> "it" thứ hai chỉ VÒNG GỌI VỐN, không phải Microsoft.
# Với nhóm này chỉ thay occurrence ĐẦU TIÊN (gần antecedent nhất, đáng tin nhất).
BARE_PRONOUNS = {"it", "its", "they", "them", "their", "theirs",
                 "he", "him", "his", "she", "her", "hers"}

def build_coref_prompt(payload):
    return f"""
For each chunk, list pronoun/generic-reference substitutions.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "substitutions": [{{"mention": "the company", "antecedent": "Microsoft"}}],
      "unresolved_mentions": ["it"]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

def apply_substitutions(text, subs):
    """Áp phép thế trong Python, khớp nguyên cụm và không phân biệt hoa thường.

    Mọi thay đổi đều truy vết được; phần văn bản còn lại byte-for-byte y nguyên.
    """
    out, applied = text, []
    for s in (subs or [])[:MAX_SUBS_PER_CHUNK]:
        if not isinstance(s, dict):
            continue
        mention = norm_space(s.get("mention"))
        antecedent = norm_space(s.get("antecedent"))
        if not mention or not antecedent:
            continue
        if mention.lower() == antecedent.lower():
            continue
        # Chặn phép thế vô lý: mention quá dài (LLM trả về cả câu) hoặc
        # antecedent không hề xuất hiện trong chunk (dấu hiệu bịa thực thể).
        if len(mention) > 40 or len(antecedent) > 60:
            continue
        if antecedent.lower() not in text.lower():
            continue
        pat = re.compile(rf"(?<!\w){re.escape(mention)}(?!\w)", re.I)
        limit = 1 if mention.lower() in BARE_PRONOUNS else 0   # 0 = thay tất cả
        new_out, n = pat.subn(antecedent, out, count=limit)
        if n:
            out = new_out
            applied.append({"mention": mention, "antecedent": antecedent,
                            "n": n, "first_only": bool(limit)})
    return out, applied

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]
    obj, _ = groq_json(COREF_SYSTEM, build_coref_prompt(payload),
                       model=BULK_MODEL, max_tokens=2048, expected_out=700)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        resolved, applied = apply_substitutions(r.text, item.get("substitutions"))
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(resolved),
            "unresolved_mentions": json.dumps(item.get("unresolved_mentions", []),
                                              ensure_ascii=False),
            "applied_subs": json.dumps(applied, ensure_ascii=False),
            "status": "OK",
        })
    return pd.DataFrame(rows)

def run_coref(chunks_subset, batch_size=None, checkpoint=COREF_CKPT):
    batch_size = int(batch_size or COREF_BATCH_SIZE)

    # --- Resume từ checkpoint: mất kết nối Colab không làm mất công đã chạy ---
    done = pd.DataFrame()
    if Path(checkpoint).exists() and not FORCE_RECOMPUTE:
        done = pd.read_csv(checkpoint)
        done = done[done.chunk_id.isin(set(chunks_subset.chunk_id))]
        print(f"♻️  Checkpoint: đã có {len(done):,}/{len(chunks_subset):,} chunk, chỉ chạy phần còn lại.")

    todo = chunks_subset[~chunks_subset.chunk_id.isin(set(done.chunk_id))] if len(done) else chunks_subset
    if todo.empty:
        return done.reset_index(drop=True)

    batches_list = [todo.iloc[i:i+batch_size] for i in range(0, len(todo), batch_size)]
    print(f"Coref: {len(todo):,} chunk -> {len(batches_list)} request "
          f"(batch={batch_size}, {LLM_WORKERS} worker, ngân sách {GROQ_DEFAULT_TPM:,} token/phút)")

    def _one(batch):
        try:
            return resolve_coref_batch(batch)
        except Exception as e:
            # Fail-safe: giữ nguyên text gốc, đánh dấu để audit được.
            return pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [json.dumps(["COREF_BATCH_FAILED"])] * len(batch),
                "applied_subs": ["[]"] * len(batch),
                "status": [f"FAILED: {type(e).__name__}"] * len(batch),
            })

    results = parallel_map(_one, batches_list, desc="Coref")
    parts = [r for r in results if isinstance(r, pd.DataFrame)]
    out = pd.concat(([done] if len(done) else []) + parts, ignore_index=True)
    out.to_csv(checkpoint, index=False)
    return out

def coref_diff_report(src_df, top_n=10):
    """Chunk nào bị coref sửa nhiều nhất -> rủi ro false coreference cao nhất.

    Đây là NGUỒN DẪN CHỨNG cho câu thuyết minh #1.
    """
    rows = []
    for r in src_df.itertuples(index=False):
        orig, res = norm_space(r.text), norm_space(getattr(r, "resolved_text", "") or "")
        subs = getattr(r, "applied_subs", "[]")
        if not res or orig == res:
            continue
        rows.append({
            "chunk_id": r.chunk_id,
            "n_subs": len(json.loads(subs)) if isinstance(subs, str) else 0,
            "applied_subs": subs,
            "similarity": round(SequenceMatcher(None, orig, res).ratio(), 3),
            "unresolved": str(getattr(r, "unresolved_mentions", ""))[:100],
            "original": orig[:220],
            "resolved": res[:220],
        })
    diff_df = pd.DataFrame(rows)
    n_unres = sum(1 for r in src_df.itertuples(index=False)
                  if str(getattr(r, "unresolved_mentions", "[]")) not in ("[]", "nan", ""))
    n_failed = int((src_df.get("status", pd.Series(dtype=str)).astype(str)
                    .str.startswith("FAILED")).sum()) if "status" in src_df else 0
    print(f"Coref: {len(diff_df)}/{len(src_df)} chunk bị sửa | "
          f"{n_unres} chunk có unresolved_mentions | {n_failed} batch lỗi")
    if diff_df.empty:
        return diff_df
    print("\n>>> Chunk bị SỬA NHIỀU NHẤT (similarity thấp = rủi ro false coref cao nhất):")
    display(diff_df.sort_values("similarity").head(top_n))
    return diff_df

_t0, _s0 = time.perf_counter(), dict(LLM_STATS)
extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
extraction_source["resolved_text"] = extraction_source["resolved_text"].fillna(extraction_source["text"])
stage_report("Coreference", _t0, _s0)

coref_diff_df = coref_diff_report(extraction_source)
coref_diff_df.to_csv(OUT_DIR / "coref_diff_report.csv", index=False)

♻️  Checkpoint: đã có 200/200 chunk, chỉ chạy phần còn lại.

⏱️  [Coreference] 0.0 phút | 0 request | 0 token (output 0) | 429 hits: 0
    Token theo model (cộng dồn, hạn ~200k/ngày mỗi model): {'openai/gpt-oss-120b': 272, 'qwen/qwen3.6-27b': 43}
Coref: 98/200 chunk bị sửa | 32 chunk có unresolved_mentions | 0 batch lỗi

>>> Chunk bị SỬA NHIỀU NHẤT (similarity thấp = rủi ro false coref cao nhất):


,chunk_id,n_subs,applied_subs,similarity,unresolved,original,resolved
5,cf095df30bfa3f72399f::c0000,2,"[{""mention"": ""It"", ""antecedent"": ""OLED display with embedded heart rate sensor"", ""n"": 1, ""first_only"": true}, {""ment...",0.461,[],Samsung unveils OLED display with embedded heart rate sensor. It is a separate module attached under the display pan...,Samsung unveils OLED display with embedded heart rate sensor. OLED display with embedded heart rate sensor is a sepa...
86,07b11f3305d87962b831::c0000,2,"[{""mention"": ""the company"", ""antecedent"": ""Adobe"", ""n"": 1, ""first_only"": false}, {""mention"": ""its"", ""antecedent"": ""A...",0.618,[],20 Big Companies That Hire Remote and Hybrid Workers. Learn more about 3M. What the company is: Adobe one of the mos...,20 Big Companies That Hire Remote and Hybrid Workers. Learn more about 3M. What Adobe is: Adobe one of the most esta...
11,8e58296629f84469cd8d::c0000,3,"[{""mention"": ""its"", ""antecedent"": ""Samsung Electronics Co. Ltd."", ""n"": 1, ""first_only"": true}, {""mention"": ""its"", ""a...",0.689,[],Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023. Samsung Electronics Co. Ltd. a world...,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023. Samsung Electronics Co. Ltd. a world...
15,968c0e4d2030e462eb2a::c0000,3,"[{""mention"": ""The tech giants"", ""antecedent"": ""Apple (NASDAQ: AAPL) and Meta Platforms (NASDAQ: META)"", ""n"": 1, ""fir...",0.772,[],2 Must-Watch Big Tech Earnings Reports. As investors try to gauge how 2023 could play out two telling earnings repor...,2 Must-Watch Big Tech Earnings Reports. As investors try to gauge how 2023 could play out two telling earnings repor...
14,3e0677727f437bb532a1::c0000,3,"[{""mention"": ""its"", ""antecedent"": ""Amazon.com''s cloud computing division"", ""n"": 1, ""first_only"": true}, {""mention"":...",0.792,[],REFILE-UPDATE 1-Amazon cloud unit to invest nearly $8 billion in Ohio. Amazon.com''s cloud computing division would ...,REFILE-UPDATE 1-Amazon cloud unit to invest nearly $8 billion in Ohio. Amazon.com''s cloud computing division would ...
30,5d162210c58b91f6a091::c0000,1,"[{""mention"": ""the Data Cloud company"", ""antecedent"": ""Snowflake"", ""n"": 1, ""first_only"": false}]",0.810,[],H2O.ai Brings H2O AI Cloud to the Snowflake Manufacturing Data Cloud. H2O.ai today announced the launch of H2O AI Cl...,H2O.ai Brings H2O AI Cloud to the Snowflake Manufacturing Data Cloud. H2O.ai today announced the launch of H2O AI Cl...
21,dd608bf3ff0214721fa9::c0000,1,"[{""mention"": ""its"", ""antecedent"": ""Google"", ""n"": 1, ""first_only"": true}]",0.846,[],DuckDuckGo founder says Google’s phone and manufacturing partnerships thwart competition. The U.S. Department of Jus...,DuckDuckGo founder says Google’s phone and manufacturing partnerships thwart competition. The U.S. Department of Jus...
64,c4dc12aae93b84da4ad2::c0000,2,"[{""mention"": ""it"", ""antecedent"": ""Starlink"", ""n"": 1, ""first_only"": true}, {""mention"": ""The company"", ""antecedent"": ""...",0.860,[],Telstra signs Starlink deal to boost internet service for rural customers. Starlink is a satellite internet service ...,Telstra signs Starlink deal to boost internet service for rural customers. Starlink is a satellite internet service ...
12,426f410b850963e21108::c0000,2,"[{""mention"": ""the “Company”"", ""antecedent"": ""Boxed Inc."", ""n"": 1, ""first_only"": false}, {""mention"": ""its"", ""antecede...",0.870,[],Boxed Launches Spresso SaaS Solutions on Google Cloud Marketplace to Provide Data-Driven .... NEW YORK Jan. 10 2023 ...,Boxed Launches Spresso SaaS Solutions on Google Cloud Marketplace to Provide Data-Driven .... NEW YORK Jan. 10 2023 ...
16,13473db603d8790352b7::c0000,2,"[{""mention"": ""the Big Techs"", ""antecedent"": ""Microsoft Google and Baidu"", ""n"": 1, ""first_only"": false}, {""mention"": ...",0.879,[],Here''s how tech giants Microsoft Google and Baidu are

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [9]:
#@title 2.1 — NER + RE extraction (song song + raw checkpoint)
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Extract EVERY relation explicitly supported by the text. Do not speculate or infer.
Every element of "items" MUST be a JSON object with keys chunk_id and relations.
Echo chunk_id back exactly as given. Keep evidence under 25 words.
Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt, model=BULK_MODEL,
                     max_tokens=3072, expected_out=900)

# --- Parsing chiu duoc output lech schema ------------------------------------
# LLM thinh thoang tra ve {"items": ["chuoi", ...]} thay vi list cua object.
# Parser PHAI song sot qua cac bien the do va ghi nhan vao bang rejected,
# khong duoc nem exception: exception o buoc nay se pha huy ket qua cua
# toan bo cac request da tra tien goi API.
def _coerce_items(obj):
    if isinstance(obj, list):
        return obj
    if not isinstance(obj, dict):
        return []
    items = obj.get("items")
    if items is None:
        return [obj] if ("relations" in obj or "chunk_id" in obj) else []
    if isinstance(items, dict):
        return [items]
    return items if isinstance(items, list) else []

def _parse_extraction(obj, meta, rejected, batch_chunk_ids=None):
    triples = []
    for item in _coerce_items(obj):
        if not isinstance(item, dict):
            rejected["ITEM_NOT_OBJECT"] += 1
            continue
        cid = item.get("chunk_id")
        if cid not in meta:
            # Batch chi co 1 chunk -> con suy ra duoc provenance mot cach an toan.
            if batch_chunk_ids and len(batch_chunk_ids) == 1 and batch_chunk_ids[0] in meta:
                cid = batch_chunk_ids[0]
            else:
                rejected["UNKNOWN_CHUNK_ID"] += 1
                continue
        rels = item.get("relations")
        if isinstance(rels, dict):
            rels = [rels]
        if not isinstance(rels, list):
            rejected["RELATIONS_NOT_LIST"] += 1
            continue
        for x in rels:
            if not isinstance(x, dict):
                rejected["RELATION_NOT_OBJECT"] += 1
                continue
            s, t = norm_space(x.get("source")), norm_space(x.get("target"))
            st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
            if not s or not t:
                rejected["EMPTY_ENDPOINT"] += 1
                continue
            if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                rejected["NODE_TYPE_NOT_ALLOWED"] += 1
                continue
            if rel not in ALLOWED_RELATIONS:
                rejected["RELATION_NOT_ALLOWED"] += 1
                continue
            try:
                conf = float(x.get("confidence") or 0.0)
            except (TypeError, ValueError):
                conf = 0.0
            triples.append({
                "source_raw": s, "source_type": st,
                "relation": rel,
                "target_raw": t, "target_type": tt,
                "source_chunk_id": cid,
                "published_date": meta[cid] or "",
                "evidence": norm_space(x.get("evidence")),
                "confidence": conf,
            })
    return triples

_raw_lock = threading.Lock()

def run_extraction(source_df, batch_size=None, checkpoint=EXTRACT_CKPT):
    """Ghi RAW RESPONSE xuong dia NGAY khi nhan duoc, roi moi parse.

    Bai hoc tu lan chay truoc: mot AttributeError trong parser da xoa sach
    ket qua cua 50 request (15 phut goi API). Tach "goi API" khoi "parse"
    nghia la sua parser sau nay ton 0 request.
    """
    batch_size = int(batch_size or EXTRACT_BATCH_SIZE)
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    raw_path = Path(checkpoint).with_suffix(".raw.jsonl")

    records, done_chunks = [], set()
    if raw_path.exists() and not FORCE_RECOMPUTE:
        for line in raw_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            records.append(rec)
            done_chunks.update(rec.get("chunk_ids", []))
        print(f"♻️  Raw checkpoint: {len(records)} response, {len(done_chunks):,} chunk đã gọi API.")

    todo = source_df[~source_df.chunk_id.isin(done_chunks)]
    if not todo.empty:
        batches_list = [todo.iloc[i:i+batch_size] for i in range(0, len(todo), batch_size)]
        print(f"NER+RE: {len(todo):,} chunk -> {len(batches_list)} request "
              f"(batch={batch_size}, {LLM_WORKERS} worker)")

        def _one(batch):
            chunk_ids = batch["chunk_id"].tolist()
            obj, _ = extract_batch(batch)
            rec = {"chunk_ids": chunk_ids, "obj": obj}
            with _raw_lock:      # ghi ngay, khong doi ca me xong
                with open(raw_path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            return rec

        results = parallel_map(_one, batches_list, desc="NER+RE")
        errors = [{"batch": i, "error": str(r)}
                  for i, r in enumerate(results) if isinstance(r, Exception)]
        records += [r for r in results if isinstance(r, dict)]
    else:
        errors = []
        print("Tất cả chunk đã có raw response — chỉ parse lại, không gọi API.")

    rejected = Counter()
    triples = []
    for rec in records:
        triples += _parse_extraction(rec.get("obj"), meta, rejected,
                                     rec.get("chunk_ids"))

    all_df = pd.DataFrame(triples)
    if not all_df.empty:
        all_df = all_df.drop_duplicates(
            subset=["source_raw", "relation", "target_raw", "source_chunk_id"]
        ).reset_index(drop=True)
    all_df.to_csv(checkpoint, index=False)
    print(f"Schema allowlist đã loại: {dict(rejected) or '{}'}")
    return all_df, pd.DataFrame(errors)

_t0, _s0 = time.perf_counter(), dict(LLM_STATS)
raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
stage_report("NER+RE extraction", _t0, _s0)

print(f"\nraw_triples_df: {len(raw_triples_df):,} triple từ {len(extraction_source):,} chunk "
      f"| batch lỗi: {len(extraction_errors_df)}")
if not raw_triples_df.empty:
    print("\nPhân bố relation:")
    display(raw_triples_df.relation.value_counts().to_frame("count"))
    print("Phân bố node type:")
    display(pd.concat([raw_triples_df.source_type, raw_triples_df.target_type])
            .value_counts().to_frame("count"))
    # Provenance phải đủ NGAY TỪ TẦNG EXTRACTION, không đợi đến lúc check Neo4j.
    missing_prov = raw_triples_df[
        raw_triples_df.source_chunk_id.fillna("").eq("")
        | raw_triples_df.published_date.fillna("").eq("")
    ]
    print(f"\nTriple thiếu provenance (bị loại trước khi nạp): {len(missing_prov):,}")
    raw_triples_df = raw_triples_df.drop(missing_prov.index).reset_index(drop=True)
    display(raw_triples_df.head(10))
else:
    raise RuntimeError("Không trích xuất được triple nào — kiểm tra BULK_MODEL và extraction_source.")

♻️  Raw checkpoint: 20 response, 200 chunk đã gọi API.
Tất cả chunk đã có raw response — chỉ parse lại, không gọi API.
Schema allowlist đã loại: {}

⏱️  [NER+RE extraction] 0.0 phút | 0 request | 0 token (output 0) | 429 hits: 0
    Token theo model (cộng dồn, hạn ~200k/ngày mỗi model): {'openai/gpt-oss-120b': 272, 'qwen/qwen3.6-27b': 43}

raw_triples_df: 306 triple từ 200 chunk | batch lỗi: 0

Phân bố relation:


,count
relation,
DEVELOPED,97
USES,67
PARTNERED_WITH,64
LEADS,28
FOUNDED,19
WORKED_AT,13
INVESTED_IN,12
ACQUIRED,6


Phân bố node type:


,count
Company,367
Technology,184
Person,61



Triple thiếu provenance (bị loại trước khi nạp): 0


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Krayden,Person,FOUNDED,Senser,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Before launching Senser in 2021 Krayden and Co-Founders Or Sadeh and Yuval Lev,1.0
1,Or Sadeh,Person,FOUNDED,Senser,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Before launching Senser in 2021 Krayden and Co-Founders Or Sadeh and Yuval Lev,1.0
2,Yuval Lev,Person,FOUNDED,Senser,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Before launching Senser in 2021 Krayden and Co-Founders Or Sadeh and Yuval Lev,1.0
3,Krayden,Person,WORKED_AT,DriveNets,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Krayden and Co-Founders Or Sadeh and Yuval Lev honed their skills as tech leaders at DriveNets,1.0
4,Or Sadeh,Person,WORKED_AT,DriveNets,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Krayden and Co-Founders Or Sadeh and Yuval Lev honed their skills as tech leaders at DriveNets,1.0
5,Yuval Lev,Person,WORKED_AT,DriveNets,Company,125b329c27f612e3fa1d::c0000,2023-09-27,Krayden and Co-Founders Or Sadeh and Yuval Lev honed their skills as tech leaders at DriveNets,1.0
6,Hywin,Company,USES,IBM Technology,Technology,4fbb6af96dd246761913::c0000,2023-04-08,Hywin Launches WealthTech Platform ... by Leveraging IBM Technology,1.0
7,Hua Zhu,Person,LEADS,Hywin,Company,4fbb6af96dd246761913::c0000,2023-04-08,Hua Zhu Hywin’s Chief Technology,1.0
8,Options Technology,Company,PARTNERED_WITH,Microsoft,Company,f73febf774301e802436::c0000,2023-06-08,Options Technology ... announced its achievement of Microsoft Solutions Partner designation,1.0
9,Kearney,Company,PARTNERED_WITH,Amazon Web Services,Company,d4134100173aefa8594b::c0000,2023-04-11,A Kearney study conducted in collaboration with Amazon Web Services,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [10]:
#@title 2.2 — Entity resolution (Vector ANN + Lexical Guard + Union-Find)
# Chỉ gồm hậu tố PHÁP LÝ. Cố tình KHÔNG đưa "ai"/"labs"/"technologies" vào đây:
# strip_suffix() so khớp trước Guard 1, nên "Meta AI" sẽ rút gọn thành "meta" và
# bị gộp thẳng vào "Meta" — đúng loại false merge mà Guard 1 sinh ra để chặn.
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    # Ticker -> tên công ty (Challenge B: ticker không bao giờ khớp lexical/vector)
    "msft": "Microsoft", "goog": "Google", "googl": "Google", "aapl": "Apple",
    "amzn": "Amazon", "nvda": "Nvidia", "tsla": "Tesla", "meta": "Meta",
    "orcl": "Oracle", "crm": "Salesforce", "adbe": "Adobe", "nflx": "Netflix",
    "intc": "Intel", "amd": "AMD", "ibm": "IBM", "baba": "Alibaba",
    # Biến thể pháp lý phổ biến
    "microsoft corp": "Microsoft", "microsoft corporation": "Microsoft",
    "google llc": "Google", "google inc": "Google",
    "meta platforms": "Meta", "meta platforms inc": "Meta",
    "apple inc": "Apple", "amazon com": "Amazon", "amazon com inc": "Amazon",
    "nvidia corporation": "Nvidia", "openai inc": "OpenAI", "open ai": "OpenAI",
}
# Quyết định có chủ đích: KHÔNG map "Alphabet" -> "Google".
# Alphabet là holding company, Google là subsidiary; gộp lại sẽ làm sai
# các quan hệ INVESTED_IN / ACQUIRED ở cấp tập đoàn. Ghi vào báo cáo.

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

# --- Lexical Guard (Challenge B) -------------------------------------------
# Ngưỡng lexical tách riêng cho Person: SequenceMatcher("sam altman","steve altman")
# = 0.727 -> với ngưỡng chung 0.72 thì HAI NGƯỜI KHÁC NHAU BỊ GỘP (false merge).
DEFAULT_MIN_RATIO = 0.72
PERSON_MIN_RATIO = 0.90

def merge_guard(a, b, typ=None):
    """Trả về (được_gộp, lý_do). Lý do được ghi thẳng vào audit table."""
    na, nb = strip_suffix(a), strip_suffix(b)
    if not na or not nb:
        return False, "GUARD_EMPTY_AFTER_SUFFIX"
    if na == nb:
        return True, "EXACT_AFTER_SUFFIX_STRIP"

    ta, tb = na.split(), nb.split()
    sa, sb = set(ta), set(tb)

    # Guard 1 — token subset: sản phẩm/đơn vị con chứa trọn tên công ty mẹ.
    # "Apple" ⊂ "Apple Watch", "Google" ⊂ "Google Cloud" -> KHÔNG gộp.
    if sa < sb or sb < sa:
        return False, "GUARD_TOKEN_SUBSET"

    # Guard 2 — Person: trùng họ nhưng khác tên là hai người khác nhau.
    # "Sam Altman" vs "Steve Altman", "Andrew Ng" vs "Alan Ng".
    if typ == "Person" and len(ta) >= 2 and len(tb) >= 2 and ta[0] != tb[0]:
        return False, "GUARD_PERSON_DIFFERENT_GIVEN_NAME"

    ratio = SequenceMatcher(None, na, nb).ratio()
    min_ratio = PERSON_MIN_RATIO if typ == "Person" else DEFAULT_MIN_RATIO
    if ratio >= min_ratio:
        return True, f"LEXICAL_OK ratio={ratio:.3f}>={min_ratio}"
    return False, f"GUARD_LEXICAL_RATIO ratio={ratio:.3f}<{min_ratio}"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, audit_floor=0.80, top_k=5):
    """threshold = ngưỡng GỘP. audit_floor = ngưỡng GHI LOG.

    Tách hai ngưỡng là chủ ý: câu thuyết minh #2 yêu cầu dẫn chứng một cặp
    similarity > 0.85 bị Guard chặn. Nếu chỉ log từ 0.90 trở lên thì mọi cặp
    trong dải 0.80–0.90 biến mất khỏi audit và không thể truy vết được.
    """
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        # Alias map chỉ áp cho Company: ticker/hậu tố pháp lý là khái niệm
        # của pháp nhân, áp nhầm sang Person/Technology sẽ tạo false merge.
        if t == "Company" and norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL",
                "reason": "Manual alias map (ticker / biến thể pháp lý)",
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                j = int(j)
                if j < 0 or i >= j:
                    continue
                score = float(score)
                if score < audit_floor:
                    continue

                if score < threshold:
                    # Ứng viên ANN nhưng dưới ngưỡng gộp -> chỉ log, không gộp.
                    audit.append({
                        "type": typ, "left": names[i], "right": names[j],
                        "similarity": score, "decision": "REJECT_LOW_SIM",
                        "reason": f"cosine={score:.3f} < threshold={threshold}",
                    })
                    continue

                ok, reason = merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": score,
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD",
                    "reason": reason,
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        if (typ, n) in mapping:
            return mapping[(typ, n)]
        if typ == "Company" and n in MANUAL_ALIASES:
            return MANUAL_ALIASES[n]
        return name

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
if len(entity_resolution_audit_df) < 10:
    # Rubric 2.3 yêu cầu >= 10 dòng audit minh bạch. Hạ NGƯỠNG GHI LOG
    # (không đổi ngưỡng gộp 0.90) để soi được cả dải ứng viên bị loại.
    print(f"Audit chỉ có {len(entity_resolution_audit_df)} dòng -> hạ audit_floor 0.80 -> 0.60.")
    entity_map, entity_resolution_audit_df = build_resolution_map(
        raw_triples_df, threshold=0.90, audit_floor=0.60, top_k=8)
triples_df = canonicalize_triples(raw_triples_df, entity_map)

n_raw = len(set(entity_map.keys()))
n_canon = len(set(entity_map.values()))
print(f"Entity resolution: {n_raw:,} mention chuẩn hoá -> {n_canon:,} entity canonical "
      f"(gộp {n_raw - n_canon:,})")
print(f"Audit rows: {len(entity_resolution_audit_df):,} "
      f"(rubric yêu cầu >= 10)\n")
if not entity_resolution_audit_df.empty:
    display(entity_resolution_audit_df.decision.value_counts().to_frame("count"))
    entity_resolution_audit_df.to_csv(AUDIT_PATH, index=False)
display(entity_resolution_audit_df.sort_values("similarity", ascending=False).head(20))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Entity resolution: 288 mention chuẩn hoá -> 281 entity canonical (gộp 7)
Audit rows: 12 (rubric yêu cầu >= 10)



,count
decision,
REJECT_LOW_SIM,6
MERGE_MANUAL,4
MERGE_VECTOR,1
REJECT_GUARD,1


,type,left,right,similarity,decision,reason
0,Company,Meta,Meta,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
1,Company,IBM,IBM,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
2,Company,Apple Inc,Apple,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
3,Company,AMD,AMD,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
9,Technology,GPT-3.5 turbo,GPT3.5-turbo,0.975751,MERGE_VECTOR,LEXICAL_OK ratio=0.870>=0.72
10,Technology,Azure,Microsoft Azure,0.960729,REJECT_GUARD,GUARD_TOKEN_SUBSET
8,Technology,Copilot,Copilot X,0.873182,REJECT_LOW_SIM,cosine=0.873 < threshold=0.9
11,Technology,GPT 3,GPT-4,0.850249,REJECT_LOW_SIM,cosine=0.850 < threshold=0.9
4,Company,Amazon.com,Amazon,0.847531,REJECT_LOW_SIM,cosine=0.848 < threshold=0.9
6,Technology,ChatGPT,ChatGPT API,0.835930,REJECT_LOW_SIM,cosine=0.836 < threshold=0.9


In [11]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    """UNWIND $rows theo batch 1000 — KHÔNG chạy MERGE từng row.

    1 round-trip cho 1000 node thay vì 1000 round-trip; trên AuraDB (RTT ~50ms)
    đây là khác biệt giữa ~1 giây và ~50 giây cho cùng lượng dữ liệu.
    """
    total = 0
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)
            total += len(b)
    print(f"Bulk insert nodes: {total:,} row (batch={batch_size})")

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    # Chặn ngay tại tầng ứng dụng: không để cạnh thiếu provenance chạm tới DB.
    bad = triples_df[
        triples_df.source_chunk_id.fillna("").eq("")
        | triples_df.published_date.fillna("").eq("")
    ]
    if len(bad):
        print(f"⚠️  Bỏ {len(bad):,} triple thiếu provenance trước khi nạp.")
        triples_df = triples_df.drop(bad.index)

    total = 0
    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)
            total += len(b)
    print(f"Bulk insert edges: {total:,} row (batch={batch_size})")

nodes_df = build_nodes(triples_df)
print(f"nodes_df: {len(nodes_df):,} entity canonical")
display(nodes_df.type.value_counts().to_frame("count"))
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

nodes_df: 286 entity canonical


,count
type,
Company,169
Technology,92
Person,25


Bulk insert nodes: 286 row (batch=1000)


Bulk insert edges: 305 row (batch=1000)


In [12]:
#@title 2.4 — Sanity checks (rubric 2.2: invalid_provenance_edges phải = 0)
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
       OR r.source_chunk_id = '' OR r.published_date = ''
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0, "❌ Có cạnh thiếu provenance — rubric trừ 5 điểm."
    print("✅ Edge provenance integrity: 100% cạnh có source_chunk_id + published_date.")

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    print("\n>>> TOP 15 DEGREE (dẫn chứng cho câu thuyết minh #3 về super-node):")
    display(top)
    top.to_csv(SUPERNODE_PATH, index=False)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 286, 'edges': 305, 'invalid_provenance_edges': 0}
✅ Edge provenance integrity: 100% cạnh có source_chunk_id + published_date.

>>> TOP 15 DEGREE (dẫn chứng cho câu thuyết minh #3 về super-node):


,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,38
1,81d12c93b121500b9963ffc1,IoT,Technology,32
2,00976a066f2bf56f1fb38754,SpaceX,Company,31
3,0188b38fd889ff606a3ab9aa,Starlink,Technology,24
4,9db24eb357b51a7992c04ce4,Elon Musk,Person,22
5,0e132222d1315530d6efca52,Google,Company,14
6,773eeb9b7cc008bff365fcdd,OpenAI,Company,12
7,68b3544368862fc263941c8d,Amazon,Company,12
8,e6c3db5c28c9a15bfafd2e04,Apple,Company,11
9,f4adaa883e92217c80dab7fb,ChatGPT,Technology,9


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [13]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)
_ctx, _docs = retrieve_flat_context("artificial intelligence investment", k=3)
display(_docs[["score","chunk_id","published_date"]])

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1501


,score,chunk_id,published_date
0,0.667783,8c53c428fb2bfa797e4f::c0000,2023-01-14
1,0.659007,97cfd826fe19f660dc7c::c0000,2023-09-18
2,0.639214,07b40253cdec1386547f::c0000,2023-09-22


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [14]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""", model=BULK_MODEL, max_tokens=512, expected_out=200)
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")
    print(f"Entity matcher: {len(entity_match_store):,} entity")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            for e in exact:
                e["match_mode"] = "EXACT"
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type,
                            "match_mode":f"FUZZY({float(sims[j]):.2f})"})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

Entity matcher: 286 entity


In [15]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100      # bậc vượt ngưỡng này thì coi là super-node
SUPER_NODE_EDGE_CAP = 50     # super-node chỉ lấy 50 cạnh mới nhất
GLOBAL_EDGE_CAP = 250        # trần tổng số cạnh trong toàn bộ context
MAX_GRAPH_CONTEXT_CHARS = 8000   # ~2500 token; giữ 1 request dưới ngân sách 8000 TPM

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    """Lấy tối đa `limit` cạnh, ưu tiên published_date mới nhất.

    ORDER BY ... LIMIT chạy Ở PHÍA DATABASE: super-node không bao giờ
    truyền hàng nghìn cạnh về client rồi mới cắt.
    """
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False, seed_ids=None):
    """seed_ids: bỏ qua LLM seed extraction và dùng thẳng danh sách node id.

    Dùng cho unit test super-node — test phải tất định và không tốn quota LLM.
    """
    if seed_ids is not None:
        seeds = [{"id": i, "name": i, "type": None, "match_mode": "INJECTED"} for i in seed_ids]
    else:
        seeds = match_seeds(query)

    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","matched_seeds":[],
                              "expanded_nodes":0,"collected_edges":0,
                              "supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        fetched = recent_edges(node_id, limit)
        if degree > SUPER_NODE_DEGREE:
            supernode_events[-1]["fetched"] = len(fetched)

        for e in fetched:
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
            "global_edge_cap_hit": len(collected) >= GLOBAL_EDGE_CAP,
        }
    }
    return out if return_debug else out["context"]

In [16]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
        "prompt_tokens": usage.get("prompt_tokens"),
    }

def answer_flat_rag(question):
    t0 = time.perf_counter()
    context, retrieved = retrieve_flat_context(question, k=6)
    retrieval_s = time.perf_counter() - t0
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved,
                "retrieval_latency_s":retrieval_s,
                "e2e_latency_s":retrieval_s + out["latency_s"]})
    return out

def answer_graph_rag(question):
    t0 = time.perf_counter()
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    retrieval_s = time.perf_counter() - t0
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs,
                "retrieval_latency_s":retrieval_s,
                "e2e_latency_s":retrieval_s + out["latency_s"]})
    return out

# --- Smoke test: bắt lỗi TRƯỚC khi chạy vòng eval đầy đủ ---
_smoke_q = f"What is {nodes_df.iloc[0]['name']} known for according to the news?"
print("Smoke question:", _smoke_q)
try:
    _sg = answer_graph_rag(_smoke_q)
except Exception as _e:
    print(f"[bo qua] smoke test — {type(_e).__name__}: {str(_e)[:120]}")
    _sg = None
if _sg:
    print("[GraphRAG diagnostics]", {k: v for k, v in _sg["graph_debug"]["diagnostics"].items()
                                    if k != "matched_seeds"})
    print("[seeds]", _sg["graph_debug"]["diagnostics"].get("matched_seeds"))
    print("[answer]", _sg["answer"][:600])

Smoke question: What is BICS known for according to the news?


   🛑 openai/gpt-oss-120b cạn TPD (196739/200000 token/ngày). Chuyển sang model dự phòng.


[GraphRAG diagnostics] {'expanded_nodes': 2, 'collected_edges': 32, 'supernode_events': [], 'global_edge_cap_hit': False}
[seeds] [{'id': '0007cc851eee989d3afe5f3f', 'name': 'BICS', 'type': 'Company', 'match_mode': 'EXACT'}]
[answer] According to the news, BICS is known for providing IoT solutions [chunk_id=42921aa66ae603026e9c::c0001].


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [17]:
#@title 4.0 — Sinh Golden Dataset TỪ ĐỒ THỊ THẬT (không bịa reference_answer)
# 4/5 câu starter gốc để reference_answer = "" và ghi TO_BE_FILLED_FROM_DATASET.
# Nếu tự bịa đáp án, LLM Judge sẽ chấm faithfulness thấp cho CẢ HAI hệ và toàn bộ
# benchmark mất ý nghĩa. Thay vào đó: truy vấn Neo4j lấy quan hệ/đường đi CÓ THẬT
# rồi sinh câu hỏi + reference_answer bám đúng bằng chứng trong corpus.

FACTOID_TEMPLATES = {
    "ACQUIRED":       ("Which company acquired {target}?", "{source} acquired {target}."),
    "FOUNDED":        ("Who or what founded {target}?", "{source} founded {target}."),
    "INVESTED_IN":    ("Which organization invested in {target}?", "{source} invested in {target}."),
    "DEVELOPED":      ("Which organization developed {target}?", "{source} developed {target}."),
    "WORKED_AT":      ("Where did {source} work?", "{source} worked at {target}."),
    "PARTNERED_WITH": ("Which organization partnered with {target}?", "{source} partnered with {target}."),
    "USES":           ("Which technology does {source} use?", "{source} uses {target}."),
    "LEADS":          ("Who leads {target}?", "{source} leads {target}."),
}

def fetch_factoid_candidates(limit=30):
    return pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    WHERE r.evidence IS NOT NULL AND r.evidence <> '' AND r.confidence >= 0.7
    RETURN a.name AS source, type(r) AS relation, b.name AS target,
           r.published_date AS date, r.source_chunk_id AS chunk, r.evidence AS evidence,
           r.confidence AS confidence
    ORDER BY r.confidence DESC, r.published_date DESC
    LIMIT $limit
    """, limit=int(limit)))

def fetch_multihop_candidates(limit=30):
    """Path 2 hop mà 2 cạnh đến từ 2 CHUNK KHÁC NHAU.

    Điều kiện r1.source_chunk_id <> r2.source_chunk_id là mấu chốt: nó đảm bảo
    câu trả lời không nằm gọn trong một chunk nào -> đúng loại câu mà Flat RAG
    top-k gặp khó và GraphRAG có lợi thế.
    """
    return pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r1]->(b:Entity)-[r2]->(c:Entity)
    WHERE a <> b AND b <> c AND a <> c
      AND r1.source_chunk_id <> r2.source_chunk_id
    RETURN a.name AS a, type(r1) AS rel1, b.name AS b, type(r2) AS rel2, c.name AS c,
           r1.published_date AS date1, r2.published_date AS date2,
           r1.source_chunk_id AS chunk1, r2.source_chunk_id AS chunk2,
           r1.evidence AS ev1, r2.evidence AS ev2
    LIMIT $limit
    """, limit=int(limit)))

def fetch_crossdoc_candidates(limit=30):
    """Cặp entity được nhắc tới trong >= 2 chunk khác nhau -> cần tổng hợp đa tài liệu."""
    return pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    WITH a, b, collect(DISTINCT r.source_chunk_id) AS chunks,
         collect(DISTINCT type(r)) AS rels,
         collect(DISTINCT r.published_date) AS dates,
         collect(r.evidence)[0..3] AS evidences
    WHERE size(chunks) >= 2
    RETURN a.name AS a, b.name AS b, rels, chunks, dates, evidences,
           size(chunks) AS n_chunks
    ORDER BY n_chunks DESC
    LIMIT $limit
    """, limit=int(limit)))

factoid_cand = fetch_factoid_candidates()
multihop_cand = fetch_multihop_candidates()
crossdoc_cand = fetch_crossdoc_candidates()

print(f"Ứng viên: factoid={len(factoid_cand)}, multi-hop={len(multihop_cand)}, cross-doc={len(crossdoc_cand)}")
print("\n>>> FACTOID candidates"); display(factoid_cand.head(8))
print(">>> MULTI-HOP candidates"); display(multihop_cand.head(8))
print(">>> CROSS-DOC candidates"); display(crossdoc_cand.head(8))

def draft_golden(n_factoid=2, n_multihop=2, n_crossdoc=2):
    rows, i = [], 0
    seen = set()   # khu trung theo cap thuc the, khong de 2 cau hoi giong nhau

    for r in factoid_cand.itertuples(index=False):
        if len([x for x in rows if x["group"] == "factoid"]) >= n_factoid:
            break
        tpl = FACTOID_TEMPLATES.get(r.relation)
        if not tpl:
            continue
        key = ("f", r.source, r.relation, r.target)
        if key in seen:
            continue
        seen.add(key)
        i += 1
        rows.append({
            "id": f"G{i:02d}", "group": "factoid",
            "question": tpl[0].format(source=r.source, target=r.target),
            "reference_answer": tpl[1].format(source=r.source, target=r.target)
                                + (f" (published {r.date})" if r.date else ""),
            "reference_evidence": f"chunk={r.chunk} | {norm_space(r.evidence)[:200]}",
        })

    for r in multihop_cand.itertuples(index=False):
        if len([x for x in rows if x["group"] == "multi-hop"]) >= n_multihop:
            break
        key = ("m", r.a, r.c)
        if key in seen:
            continue
        seen.add(key)
        i += 1
        rows.append({
            "id": f"G{i:02d}", "group": "multi-hop",
            "question": (f"According to the news corpus, {r.a} is connected to {r.c} "
                         f"through one intermediary entity. Identify the intermediary "
                         f"and state both relationships with their dates."),
            "reference_answer": (f"The intermediary is {r.b}. {r.a} -{r.rel1}-> {r.b} "
                                 f"(published {r.date1 or 'unknown'}), and {r.b} -{r.rel2}-> {r.c} "
                                 f"(published {r.date2 or 'unknown'})."),
            "reference_evidence": (f"chunk1={r.chunk1} | {norm_space(r.ev1)[:120]} || "
                                   f"chunk2={r.chunk2} | {norm_space(r.ev2)[:120]}"),
        })

    for r in crossdoc_cand.itertuples(index=False):
        if len([x for x in rows if x["group"] == "cross-doc"]) >= n_crossdoc:
            break
        key = ("c", tuple(sorted([r.a, r.b])))
        if key in seen:
            continue
        seen.add(key)
        i += 1
        rels = ", ".join(r.rels)
        dates = ", ".join([d for d in r.dates if d]) or "unknown"
        rows.append({
            "id": f"G{i:02d}", "group": "cross-doc",
            "question": (f"Multiple articles mention both {r.a} and {r.b}. Summarize what "
                         f"the corpus reports about their relationship across those articles, "
                         f"including the relationship types and publication dates."),
            "reference_answer": (f"Across {r.n_chunks} distinct source chunks the corpus links "
                                 f"{r.a} to {r.b} via: {rels}. Reported dates: {dates}."),
            "reference_evidence": "chunks=" + "; ".join(r.chunks[:4]),
        })

    return pd.DataFrame(rows)

if Path(GOLDEN_PATH).exists():
    golden_draft_df = pd.read_csv(GOLDEN_PATH)
    print(f"\n📄 Đã có {GOLDEN_PATH} — dùng lại (xoá file nếu muốn sinh lại).")
else:
    golden_draft_df = draft_golden()
    golden_draft_df.to_csv(GOLDEN_PATH, index=False)
    print(f"\n✅ Đã sinh {len(golden_draft_df)} câu golden -> {GOLDEN_PATH}")

display(golden_draft_df)
print("""
👉 VIỆC THỦ CÔNG BẮT BUỘC:
   Đọc lại từng dòng ở trên. Câu hỏi/đáp án được sinh máy móc từ triple, có thể
   ngô nghê hoặc dựa trên triple sai do LLM extract nhầm. Hãy:
     1. Sửa câu chữ cho tự nhiên,
     2. Đối chiếu reference_evidence với chunk gốc trong chunks_df,
     3. Loại/thay câu nào dựa trên triple sai,
   rồi ghi đè file bằng cell 4.0b bên dưới.
""")

Ứng viên: factoid=30, multi-hop=30, cross-doc=16

>>> FACTOID candidates


,source,relation,target,date,chunk,evidence,confidence
0,X,DEVELOPED,audio video calling,2023-10-25,e23358a050b9d0082016::c0000,Social media platform X ... is launching an early version of video and audio calling,1.0
1,Elon Musk,LEADS,X,2023-10-25,e23358a050b9d0082016::c0000,chief technology officer and owner Elon Musk,1.0
2,PopMenu,PARTNERED_WITH,OpenTable,2023-10-20,1e6eaab64a18a0443f70::c0000,PopMenu integrating with OpenTable,1.0
3,Uber Direct,PARTNERED_WITH,Deliverect,2023-10-20,1e6eaab64a18a0443f70::c0000,Uber Direct partnering with Deliverect,1.0
4,Soundhound,PARTNERED_WITH,Olo,2023-10-20,1e6eaab64a18a0443f70::c0000,Soundhound partnering with Olo,1.0
5,Tencent Games,USES,Brompton Technology,2023-10-18,5719f1a360e208cb68d8::c0000,Tencent Games’ CDD Virtual Studio is powered by Brompton Technology,1.0
6,SpaceX,DEVELOPED,Starlink,2023-10-13,212ca3357c1aee243caf::c0000,SpaceX's Starlink set to launch satellite phone service in 2024.,1.0
7,Intel,DEVELOPED,Arc GPU Driver,2023-10-12,ce496b595b2fbcd99947::c0000,Intel Arc GPU Driver Pilots A Massive 159% Uplift,1.0


>>> MULTI-HOP candidates


,a,rel1,b,rel2,c,date1,date2,chunk1,chunk2,ev1,ev2
0,Elon Musk,FOUNDED,SpaceX,DEVELOPED,Starlink,2023-07-27,2023-01-12,43168210f03cb0a63374::c0000,b2e1a10cdad1061977d7::c0000,SpaceX founded by Elon Musk,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
1,Elon Musk,FOUNDED,SpaceX,DEVELOPED,Starlink,2023-10-07,2023-01-12,b73a055f840c48fbb9b2::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk’s SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
2,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-07-07,2023-01-12,52cf68fa2f804a20e829::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk’s SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
3,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-07-07,2023-01-12,5f625b8a93a120c23ecb::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk’s SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
4,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-07-07,2023-01-12,f79de83542ec73932873::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk's SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
5,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-01-09,2023-01-12,7ec0e2da5a7dc0cf7f30::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk''s aerospace company SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
6,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-02-22,2023-01-12,41ace907dc5bb79b9796::c0000,b2e1a10cdad1061977d7::c0000,Elon Musk''s SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service
7,Elon Musk,LEADS,SpaceX,DEVELOPED,Starlink,2023-02-03,2023-01-12,c29a5a7717a9f4201890::c0000,b2e1a10cdad1061977d7::c0000,the satellite internet unit of Elon Musk''s SpaceX,Elon Musk''s Company Powered by SpaceX... Try any Starlink service


>>> CROSS-DOC candidates


,a,b,rels,chunks,dates,evidences,n_chunks
0,SpaceX,Starlink,[DEVELOPED],"[b2e1a10cdad1061977d7::c0000, 43168210f03cb0a63374::c0000, 31b65821b712cc0ca3b2::c0000, bb3959bf4974afb4ab9c::c0000,...","[2023-01-12, 2023-07-27, 2023-10-07, 2023-10-11, 2023-10-08, 2023-07-07, 2022-11-30, 2023-01-09, 2023-02-22, 2023-10...","[Elon Musk''s Company Powered by SpaceX... Try any Starlink service, satellite internet service called Starlink, Spa...",16
1,Elon Musk,SpaceX,"[FOUNDED, LEADS]","[b2e1a10cdad1061977d7::c0000, 43168210f03cb0a63374::c0000, b73a055f840c48fbb9b2::c0000, 52cf68fa2f804a20e829::c0000,...","[2023-01-12, 2023-07-27, 2023-10-07, 2023-07-07, 2023-01-09, 2023-02-22, 2023-02-03, 2023-01-14, 2023-01-13]","[Elon Musk''s Company Powered by SpaceX, SpaceX founded by Elon Musk, Elon Musk’s SpaceX]",13
2,Jeff Bezos,Blue Origin,[FOUNDED],"[b73a055f840c48fbb9b2::c0000, 31b65821b712cc0ca3b2::c0000, 1204d4e596b20ee8a4cf::c0000, 290f7d442d055b3669c0::c0000,...","[2023-10-07, 2023-10-08, 2023-10-11]","[Jeff Bezos who has his own rocket company, who has his own rocket company Blue Origin, who has his own rocket compa...",6
3,Amazon,test satellites,[DEVELOPED],"[b73a055f840c48fbb9b2::c0000, 31b65821b712cc0ca3b2::c0000, 1204d4e596b20ee8a4cf::c0000, 290f7d442d055b3669c0::c0000]","[2023-10-07, 2023-10-08, 2023-10-11]","[Amazon launched the first test satellites, Amazon launched the first test satellites, Amazon launched the first tes...",4
4,OpenAI,ChatGPT,[DEVELOPED],"[eb4b5c00ae89793fab09::c0000, 224c0f1055b396927221::c0000, ae77d4dcd7d66b6ebfb4::c0000, b8dbf3e22497d322ffdc::c0000]","[2023-06-15, 2023-01-20, 2023-09-10, 2023-03-23]","[launch of ChatGPT last fall, ChatGPT a large language model (LLM) developed by OpenAI, startup introduced ChatGPT]",4
5,KPMG,Microsoft,"[INVESTED_IN, PARTNERED_WITH]","[35f9e232b3c44ebab014::c0000, 3dd6936aea051cd1e68c::c0000, 5e28d48f0b306d254eca::c0000]","[2023-07-11, 2023-07-14, 2023-07-16]","[KPMG will spend $2 billion ... through an expanded partnership with Microsoft, KPMG has pledged a $2 billion invest...",3
6,Elon Musk,Twitter,"[ACQUIRED, INVESTED_IN, LEADS]","[250939bab78cf84668d6::c0000, 26caff5622d8bee2daa0::c0000, 5fc1e154ab293d8b01ab::c0000]","[2023-03-17, 2023-02-05, 2023-07-03]","[Elon Musk-owned Twitter, Elon Musk continues to look for ways to make Twitter profitable and make back Elon Musk in...",3
7,Elon Musk,Starlink,[LEADS],"[e48cc45d32942970e27d::c0000, 95d38b12e8b6acd4ba62::c0000, 502da577c03c28da9743::c0000]","[2023-07-15, 2023-03-27, 2022-12-27]","[Elon Musk's Starlink satellites, Musk''s Starlink''s V2 sats experience de-orbit-worthy ''issues'', Elon Musk’s sat...",3



📄 Đã có d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\data\golden_dataset.csv — dùng lại (xoá file nếu muốn sinh lại).


,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Which organization developed audio video calling?,X developed audio video calling. (published 2023-10-25),chunk=e23358a050b9d0082016::c0000 | Social media platform X ... is launching an early version of video and audio cal...
1,G02,factoid,Who leads X?,Elon Musk leads X. (published 2023-10-25),chunk=e23358a050b9d0082016::c0000 | chief technology officer and owner Elon Musk
2,G03,multi-hop,"According to the news corpus, Elon Musk is connected to Starlink through one intermediary entity. Identify the inter...","The intermediary is SpaceX. Elon Musk -FOUNDED-> SpaceX (published 2023-07-27), and SpaceX -DEVELOPED-> Starlink (pu...",chunk1=43168210f03cb0a63374::c0000 | SpaceX founded by Elon Musk || chunk2=b2e1a10cdad1061977d7::c0000 | Elon Musk''...
3,G04,cross-doc,Multiple articles mention both SpaceX and Starlink. Summarize what the corpus reports about their relationship acros...,"Across 16 distinct source chunks the corpus links SpaceX to Starlink via: DEVELOPED. Reported dates: 2023-01-12, 202...",chunks=b2e1a10cdad1061977d7::c0000; 43168210f03cb0a63374::c0000; 31b65821b712cc0ca3b2::c0000; bb3959bf4974afb4ab9c::...
4,G05,cross-doc,Multiple articles mention both Elon Musk and SpaceX. Summarize what the corpus reports about their relationship acro...,"Across 13 distinct source chunks the corpus links Elon Musk to SpaceX via: FOUNDED, LEADS. Reported dates: 2023-01-1...",chunks=b2e1a10cdad1061977d7::c0000; 43168210f03cb0a63374::c0000; b73a055f840c48fbb9b2::c0000; 52cf68fa2f804a20e829::...



👉 VIỆC THỦ CÔNG BẮT BUỘC:
   Đọc lại từng dòng ở trên. Câu hỏi/đáp án được sinh máy móc từ triple, có thể
   ngô nghê hoặc dựa trên triple sai do LLM extract nhầm. Hãy:
     1. Sửa câu chữ cho tự nhiên,
     2. Đối chiếu reference_evidence với chunk gốc trong chunks_df,
     3. Loại/thay câu nào dựa trên triple sai,
   rồi ghi đè file bằng cell 4.0b bên dưới.



In [18]:
#@title 4.1 — Golden Dataset: chỉnh tay + validate
# Sửa trực tiếp trong OVERRIDES rồi chạy lại cell. Key = id câu hỏi.
# Để {} nếu chấp nhận nguyên bản draft sinh từ đồ thị ở cell 4.0.
OVERRIDES = {
    # "G01": {"question": "...", "reference_answer": "...", "reference_evidence": "..."},
}

golden_df = pd.read_csv(GOLDEN_PATH)
for _id, _patch in OVERRIDES.items():
    _m = golden_df.id == _id
    if not _m.any():
        print(f"⚠️  Không thấy id {_id} trong golden dataset.")
        continue
    for _c, _v in _patch.items():
        golden_df.loc[_m, _c] = _v
golden_df.to_csv(GOLDEN_PATH, index=False)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")

    groups = set(df.group.unique())
    missing_groups = {"factoid","multi-hop","cross-doc"} - groups
    if missing_groups:
        raise ValueError(f"Thiếu nhóm câu hỏi: {missing_groups} (rubric 3.1 yêu cầu đủ 3 nhóm).")
    if len(df) < 5:
        raise ValueError(f"Chỉ có {len(df)} câu — rubric yêu cầu tối thiểu 5.")

    empty = df[df.reference_answer.fillna("").str.strip().eq("")]
    if require_answers and len(empty):
        display(empty[["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print(f"✅ Golden Dataset valid: {len(df)} câu | nhóm = {sorted(groups)}")

display(golden_df)
display(golden_df.group.value_counts().to_frame("count"))
validate_golden(golden_df, require_answers=True)

,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Which organization developed audio video calling?,X developed audio video calling. (published 2023-10-25),chunk=e23358a050b9d0082016::c0000 | Social media platform X ... is launching an early version of video and audio cal...
1,G02,factoid,Who leads X?,Elon Musk leads X. (published 2023-10-25),chunk=e23358a050b9d0082016::c0000 | chief technology officer and owner Elon Musk
2,G03,multi-hop,"According to the news corpus, Elon Musk is connected to Starlink through one intermediary entity. Identify the inter...","The intermediary is SpaceX. Elon Musk -FOUNDED-> SpaceX (published 2023-07-27), and SpaceX -DEVELOPED-> Starlink (pu...",chunk1=43168210f03cb0a63374::c0000 | SpaceX founded by Elon Musk || chunk2=b2e1a10cdad1061977d7::c0000 | Elon Musk''...
3,G04,cross-doc,Multiple articles mention both SpaceX and Starlink. Summarize what the corpus reports about their relationship acros...,"Across 16 distinct source chunks the corpus links SpaceX to Starlink via: DEVELOPED. Reported dates: 2023-01-12, 202...",chunks=b2e1a10cdad1061977d7::c0000; 43168210f03cb0a63374::c0000; 31b65821b712cc0ca3b2::c0000; bb3959bf4974afb4ab9c::...
4,G05,cross-doc,Multiple articles mention both Elon Musk and SpaceX. Summarize what the corpus reports about their relationship acro...,"Across 13 distinct source chunks the corpus links Elon Musk to SpaceX via: FOUNDED, LEADS. Reported dates: 2023-01-1...",chunks=b2e1a10cdad1061977d7::c0000; 43168210f03cb0a63374::c0000; b73a055f840c48fbb9b2::c0000; 52cf68fa2f804a20e829::...


,count
group,
factoid,2
cross-doc,2
multi-hop,1


✅ Golden Dataset valid: 5 câu | nhóm = ['cross-doc', 'factoid', 'multi-hop']


In [19]:
#@title 4.2 — LLM-as-a-Judge
# JUDGE_MODEL phải KHÁC dòng với GROQ_MODEL. Nếu judge và generator là cùng một
# model, nó sẽ ưu ái chính output của mình (self-preference bias) và điểm
# faithfulness/comprehensiveness của cả hai hệ đều bị thổi lên -> benchmark vô nghĩa.
if JUDGE_PROVIDER == "groq" and JUDGE_MODEL == GROQ_MODEL:
    print(f"⚠️  CẢNH BÁO SELF-BIAS: JUDGE_MODEL == GROQ_MODEL == {GROQ_MODEL}. "
          f"Hãy đổi JUDGE_MODEL sang model khác dòng trong Colab Secrets.")
else:
    print(f"✅ Judge = {JUDGE_PROVIDER}:{JUDGE_MODEL} | Generator = groq:{GROQ_MODEL}")

JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:9000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        try:
            out[k] = max(1, min(5, int(float(obj.get(k, 1)))))
        except (TypeError, ValueError):
            out[k] = 1
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

✅ Judge = groq:qwen/qwen3.6-27b | Generator = groq:openai/gpt-oss-120b


In [20]:
#@title 4.3 — Evaluation runner + checkpoint
def run_evaluation(golden_df, checkpoint=CHECKPOINT):
    # Resume: cau nao da co trong checkpoint thi khong goi lai LLM.
    rows, done = [], set()
    if Path(checkpoint).exists() and not FORCE_RECOMPUTE:
        prev = pd.read_csv(checkpoint)
        prev = prev[prev.id.isin(set(golden_df.id))]
        rows = prev.to_dict("records")
        done = set(prev.id)
        print(f"resume eval: {len(done)}/{len(golden_df)} cau da cham, bo qua.")
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        if q.id in done:
            continue
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        diag = graph["graph_debug"]["diagnostics"]
        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":round(flat["e2e_latency_s"], 3),
            "graph_latency_s":round(graph["e2e_latency_s"], 3),
            "flat_retrieval_s":round(flat["retrieval_latency_s"], 3),
            "graph_retrieval_s":round(graph["retrieval_latency_s"], 3),
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            # --- Trường phục vụ FAILURE ANALYSIS (root-cause) ---
            "graph_matched_seeds":"; ".join(
                f"{s.get('name')}[{s.get('match_mode')}]" for s in diag.get("matched_seeds", [])
            ) or "NO_SEED",
            "graph_expanded_nodes":diag.get("expanded_nodes", 0),
            "graph_collected_edges":diag.get("collected_edges", 0),
            "graph_supernode_events":len(diag.get("supernode_events", [])),
            "graph_global_cap_hit":diag.get("global_edge_cap_hit", False),
            "flat_retrieved_chunks":"; ".join(flat["retrieved"].chunk_id.tolist())
                if not flat["retrieved"].empty else "",
            "flat_context":flat["context"],
            "graph_context":graph["context"],
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

_stats_before = dict(LLM_STATS)
eval_results_df = run_evaluation(golden_df)
print(f"\nLLM calls dùng cho evaluation: {LLM_STATS['calls'] - _stats_before['calls']} "
      f"| tokens: {LLM_STATS['total_tokens'] - _stats_before['total_tokens']:,}")
display(eval_results_df[[
    "id","group",
    "flat_comprehensiveness","graph_comprehensiveness",
    "flat_faithfulness","graph_faithfulness",
    "flat_multi_hop_reasoning","graph_multi_hop_reasoning",
    "flat_latency_s","graph_latency_s",
    "flat_total_tokens","graph_total_tokens",
    "graph_matched_seeds","graph_collected_edges",
]])

resume eval: 5/5 cau da cham, bo qua.

Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]


LLM calls dùng cho evaluation: 0 | tokens: 0


,id,group,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,graph_matched_seeds,graph_collected_edges
0,G01,factoid,5,5,5,5,5,5,0.361,0.937,731,681,audio video calling[EXACT],2
1,G02,factoid,5,5,5,5,5,5,57.082,1.397,996,2329,X[EXACT],23
2,G03,multi-hop,4,5,5,5,4,5,45.989,3.678,1166,4138,Elon Musk[EXACT]; Starlink[EXACT],84
3,G04,cross-doc,1,2,1,4,1,3,57.335,3.722,1080,4275,SpaceX[EXACT]; Starlink[EXACT],82
4,G05,cross-doc,1,5,1,5,1,5,57.267,3.923,1203,4548,Elon Musk[EXACT]; SpaceX[EXACT],47


In [21]:
#@title 4.4 — Comparison table + export
METRIC_MAP = {
    "Comprehensiveness":  ("flat_comprehensiveness", "graph_comprehensiveness"),
    "Faithfulness":       ("flat_faithfulness", "graph_faithfulness"),
    "Multi-hop reasoning":("flat_multi_hop_reasoning", "graph_multi_hop_reasoning"),
    "Latency (s)":        ("flat_latency_s", "graph_latency_s"),
    "Token usage":        ("flat_total_tokens", "graph_total_tokens"),
}

def _comment(metric, f, g):
    if pd.isna(f) or pd.isna(g):
        return "Không đủ dữ liệu."
    if metric in {"Latency (s)", "Token usage"}:
        if f == 0:
            return "Không đo được."
        ratio = g / f
        if ratio > 1.15:
            return f"GraphRAG đắt hơn {ratio:.2f}x — chi phí của traversal + graph context."
        if ratio < 0.85:
            return f"GraphRAG rẻ hơn {1/ratio:.2f}x — subgraph cô đọng hơn top-k chunk."
        return "Chi phí hai bên tương đương."
    delta = g - f
    if delta >= 0.75:
        return f"GraphRAG hơn {delta:+.2f} điểm — kiểm tra rationale & provenance để xác nhận."
    if delta <= -0.5:
        return f"Flat RAG hơn {-delta:.2f} điểm — graph extraction/retrieval đang mất hoặc nhiễu thông tin."
    return f"Hai phương pháp gần nhau ({delta:+.2f})."

def comparison_table(eval_df):
    rows = []
    # Từng nhóm câu hỏi + một khối tổng hợp ALL để có con số toàn cục cho báo cáo.
    parts = [(g_name, g) for g_name, g in eval_df.groupby("group")]
    parts.append(("ALL", eval_df))

    for group, g in parts:
        for metric, (fc, gc) in METRIC_MAP.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()
            rows.append({
                "Loại câu hỏi": group,
                "n": len(g),
                "Metric": metric,
                "Flat RAG": round(f, 3) if pd.notna(f) else np.nan,
                "GraphRAG": round(gr, 3) if pd.notna(gr) else np.nan,
                "Delta": round(gr - f, 3) if pd.notna(f) and pd.notna(gr) else np.nan,
                "Nhận xét phân tích": _comment(metric, f, gr),
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)

eval_results_df.to_csv(EVAL_RESULTS_PATH, index=False)
comparison_df.to_csv(SUMMARY_PATH, index=False)

# RUBRIC 3.3 ghi "reports/", README ghi "outputs/". Ghi cả hai nơi để không mất điểm.
REPORT_DIR = BASE_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for _src in (EVAL_RESULTS_PATH, SUMMARY_PATH):
    shutil.copy(_src, REPORT_DIR / Path(_src).name)

print(f"✅ Export:\n  {EVAL_RESULTS_PATH}\n  {SUMMARY_PATH}\n  + bản sao trong {REPORT_DIR}")
print(f"\nTổng chi phí LLM toàn pipeline: {LLM_STATS}")

,Loại câu hỏi,n,Metric,Flat RAG,GraphRAG,Delta,Nhận xét phân tích
0,cross-doc,2,Comprehensiveness,1.000,3.500,2.500,GraphRAG hơn +2.50 điểm — kiểm tra rationale & provenance để xác nhận.
1,cross-doc,2,Faithfulness,1.000,4.500,3.500,GraphRAG hơn +3.50 điểm — kiểm tra rationale & provenance để xác nhận.
2,cross-doc,2,Multi-hop reasoning,1.000,4.000,3.000,GraphRAG hơn +3.00 điểm — kiểm tra rationale & provenance để xác nhận.
3,cross-doc,2,Latency (s),57.301,3.822,-53.479,GraphRAG rẻ hơn 14.99x — subgraph cô đọng hơn top-k chunk.
4,cross-doc,2,Token usage,1141.500,4411.500,3270.000,GraphRAG đắt hơn 3.86x — chi phí của traversal + graph context.
5,factoid,2,Comprehensiveness,5.000,5.000,0.000,Hai phương pháp gần nhau (+0.00).
6,factoid,2,Faithfulness,5.000,5.000,0.000,Hai phương pháp gần nhau (+0.00).
7,factoid,2,Multi-hop reasoning,5.000,5.000,0.000,Hai phương pháp gần nhau (+0.00).
8,factoid,2,Latency (s),28.722,1.167,-27.554,GraphRAG rẻ hơn 24.61x — subgraph cô đọng hơn top-k chunk.
9,factoid,2,Token usage,863.500,1505.000,641.500,GraphRAG đắt hơn 1.74x — chi phí của traversal + graph context.


✅ Export:
  d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\outputs\graphrag_eval_results.csv
  d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\outputs\graphrag_vs_flatrag_summary.csv
  + bản sao trong d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\reports



Tổng chi phí LLM toàn pipeline: {'calls': 5, 'prompt_tokens': 3167, 'completion_tokens': 162, 'total_tokens': 3329, 'retries': 0, 'rate_limited': 0}


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [22]:
#@title 5.1 — Super-node check + entity audit
def degree_table(limit=15):
    return pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT $limit
    """, limit=int(limit)))

def test_supernode_policy():
    """Chứng minh cơ chế cắt tỉa super-node THỰC SỰ chạy, kể cả khi đồ thị lab nhỏ.

    Vấn đề: với EXTRACTION_MAX_CHUNKS = 400, gần như chắc chắn không node nào
    đạt degree > 100. Bản test gốc khi đó rơi vào nhánh else và không assert gì —
    tức là không có bằng chứng nào cho rubric 2.1.

    Cách xử lý: chạy 2 tầng.
      A) Kiểm tra trên ngưỡng THẬT (nếu đồ thị có super-node thật).
      B) Hạ ngưỡng TẠM THỜI xuống dưới bậc lớn nhất quan sát được để kích hoạt
         đúng nhánh code cắt tỉa, rồi khôi phục hằng số. Cùng một code path,
         chỉ khác tham số -> vẫn là bằng chứng hợp lệ cho chính sách.
    """
    global SUPER_NODE_DEGREE, SUPER_NODE_EDGE_CAP

    top = degree_table(15)
    if top.empty or top.degree.max() == 0:
        print("❌ Graph rỗng — không kiểm tra được.")
        return top, []

    print(">>> TOP 15 DEGREE:")
    display(top)
    hub = top.iloc[0]
    max_degree = int(hub.degree)
    print(f"\nNode bậc cao nhất: {hub['name']} [{hub['type']}] degree={max_degree}")
    print(f"Ngưỡng cấu hình: SUPER_NODE_DEGREE={SUPER_NODE_DEGREE}, "
          f"SUPER_NODE_EDGE_CAP={SUPER_NODE_EDGE_CAP}, GLOBAL_EDGE_CAP={GLOBAL_EDGE_CAP}")

    results = []

    # --- A. Cắt tỉa ở tầng Cypher: LIMIT chạy phía DB, có ORDER BY date DESC ---
    uncapped = recent_edges(hub["id"], 100000)
    capped = recent_edges(hub["id"], SUPER_NODE_EDGE_CAP)
    assert len(capped) <= SUPER_NODE_EDGE_CAP, "Cypher LIMIT không có tác dụng!"
    dates_capped = [e.get("published_date") or "" for e in capped]
    assert dates_capped == sorted(dates_capped, reverse=True), "Cạnh không được sắp xếp theo date DESC!"
    print(f"\n[A] recent_edges: uncapped={len(uncapped)} -> capped={len(capped)} "
          f"(<= {SUPER_NODE_EDGE_CAP}) ✅ và đã sắp xếp published_date DESC ✅")
    results.append({"check": "cypher_limit_and_recency", "uncapped": len(uncapped),
                    "capped": len(capped), "passed": True})

    # --- B. Kích hoạt nhánh super-node trong BFS ---
    real_supernode = max_degree > SUPER_NODE_DEGREE
    orig_deg, orig_cap = SUPER_NODE_DEGREE, SUPER_NODE_EDGE_CAP
    try:
        if not real_supernode:
            SUPER_NODE_DEGREE = max(0, max_degree - 1)
            SUPER_NODE_EDGE_CAP = 5
            print(f"\n[B] Đồ thị lab không có node degree > {orig_deg}. "
                  f"Hạ TẠM ngưỡng xuống degree>{SUPER_NODE_DEGREE}, cap={SUPER_NODE_EDGE_CAP} "
                  f"để kích hoạt đúng nhánh cắt tỉa.")
        else:
            print(f"\n[B] Đồ thị CÓ super-node thật (degree={max_degree} > {SUPER_NODE_DEGREE}).")

        # seed_ids -> không gọi LLM, test tất định.
        g = retrieve_graph_context("supernode policy test", max_hops=2,
                                   edge_limit=50, return_debug=True,
                                   seed_ids=[hub["id"]])
        events = g["diagnostics"]["supernode_events"]
        assert events, "Không có supernode_event nào được ghi nhận — nhánh cắt tỉa không chạy!"
        for ev in events:
            assert ev["limit"] <= SUPER_NODE_EDGE_CAP, ev
            assert ev.get("fetched", 0) <= SUPER_NODE_EDGE_CAP, ev
        assert g["diagnostics"]["collected_edges"] <= GLOBAL_EDGE_CAP

        print(f"    supernode_events = {events}")
        print(f"    collected_edges  = {g['diagnostics']['collected_edges']} "
              f"(<= GLOBAL_EDGE_CAP={GLOBAL_EDGE_CAP})")
        print(f"    graph context    = {len(g['context']):,} ký tự "
              f"(<= MAX_GRAPH_CONTEXT_CHARS={MAX_GRAPH_CONTEXT_CHARS:,})")
        assert len(g["context"]) <= MAX_GRAPH_CONTEXT_CHARS
        print("✅ Super-node cap OK: node vượt ngưỡng chỉ expand tối đa "
              f"{SUPER_NODE_EDGE_CAP} cạnh mới nhất.")
        results.append({"check": "bfs_supernode_branch", "events": len(events),
                        "collected_edges": g["diagnostics"]["collected_edges"],
                        "demo_mode": not real_supernode, "passed": True})
    finally:
        SUPER_NODE_DEGREE, SUPER_NODE_EDGE_CAP = orig_deg, orig_cap
        print(f"\n(Đã khôi phục SUPER_NODE_DEGREE={SUPER_NODE_DEGREE}, "
              f"SUPER_NODE_EDGE_CAP={SUPER_NODE_EDGE_CAP})")

    return top, results

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    print(f"Tổng audit rows: {len(audit_df):,}")
    display(audit_df.decision.value_counts().to_frame("count"))

    print("\n>>> Top similarity (mọi quyết định):")
    display(audit_df.sort_values("similarity", ascending=False).head(20))

    print("\n>>> DẪN CHỨNG CHO CÂU THUYẾT MINH #2 — "
          "cặp similarity > 0.85 nhưng bị Lexical Guard CHẶN:")
    hi_reject = audit_df[
        (audit_df.decision == "REJECT_GUARD") & (audit_df.similarity > 0.85)
    ].sort_values("similarity", ascending=False)
    if hi_reject.empty:
        print("(Không có. Xem toàn bộ REJECT_GUARD bên dưới thay thế.)")
        display(audit_df[audit_df.decision == "REJECT_GUARD"]
                .sort_values("similarity", ascending=False).head(20))
    else:
        display(hi_reject.head(20))

top_degree_df, supernode_test_results = test_supernode_policy()
top_degree_df.to_csv(SUPERNODE_PATH, index=False)
print("\n" + "=" * 70 + "\n")
show_resolution_audit(entity_resolution_audit_df)

>>> TOP 15 DEGREE:

,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,38
1,81d12c93b121500b9963ffc1,IoT,Technology,32
2,00976a066f2bf56f1fb38754,SpaceX,Company,31
3,0188b38fd889ff606a3ab9aa,Starlink,Technology,24
4,9db24eb357b51a7992c04ce4,Elon Musk,Person,22
5,0e132222d1315530d6efca52,Google,Company,14
6,773eeb9b7cc008bff365fcdd,OpenAI,Company,12
7,68b3544368862fc263941c8d,Amazon,Company,12
8,e6c3db5c28c9a15bfafd2e04,Apple,Company,11
9,f4adaa883e92217c80dab7fb,ChatGPT,Technology,9



Node bậc cao nhất: Microsoft [Company] degree=38
Ngưỡng cấu hình: SUPER_NODE_DEGREE=100, SUPER_NODE_EDGE_CAP=50, GLOBAL_EDGE_CAP=250

[A] recent_edges: uncapped=38 -> capped=38 (<= 50) ✅ và đã sắp xếp published_date DESC ✅

[B] Đồ thị lab không có node degree > 100. Hạ TẠM ngưỡng xuống degree>37, cap=5 để kích hoạt đúng nhánh cắt tỉa.


    supernode_events = [{'node_id': 'fb0f4df56fab164ec48722f0', 'degree': 38, 'limit': 5, 'fetched': 5}]
    collected_edges  = 16 (<= GLOBAL_EDGE_CAP=250)
    graph context    = 2,809 ký tự (<= MAX_GRAPH_CONTEXT_CHARS=8,000)
✅ Super-node cap OK: node vượt ngưỡng chỉ expand tối đa 5 cạnh mới nhất.

(Đã khôi phục SUPER_NODE_DEGREE=100, SUPER_NODE_EDGE_CAP=50)


Tổng audit rows: 12


,count
decision,
REJECT_LOW_SIM,6
MERGE_MANUAL,4
MERGE_VECTOR,1
REJECT_GUARD,1



>>> Top similarity (mọi quyết định):


,type,left,right,similarity,decision,reason
0,Company,Meta,Meta,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
1,Company,IBM,IBM,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
2,Company,Apple Inc,Apple,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
3,Company,AMD,AMD,1.000000,MERGE_MANUAL,Manual alias map (ticker / biến thể pháp lý)
9,Technology,GPT-3.5 turbo,GPT3.5-turbo,0.975751,MERGE_VECTOR,LEXICAL_OK ratio=0.870>=0.72
10,Technology,Azure,Microsoft Azure,0.960729,REJECT_GUARD,GUARD_TOKEN_SUBSET
8,Technology,Copilot,Copilot X,0.873182,REJECT_LOW_SIM,cosine=0.873 < threshold=0.9
11,Technology,GPT 3,GPT-4,0.850249,REJECT_LOW_SIM,cosine=0.850 < threshold=0.9
4,Company,Amazon.com,Amazon,0.847531,REJECT_LOW_SIM,cosine=0.848 < threshold=0.9
6,Technology,ChatGPT,ChatGPT API,0.835930,REJECT_LOW_SIM,cosine=0.836 < threshold=0.9



>>> DẪN CHỨNG CHO CÂU THUYẾT MINH #2 — cặp similarity > 0.85 nhưng bị Lexical Guard CHẶN:


,type,left,right,similarity,decision,reason
10,Technology,Azure,Microsoft Azure,0.960729,REJECT_GUARD,GUARD_TOKEN_SUBSET


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [23]:
#@title Bonus A — Global Search via Community Reports (NetworkX fallback)
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    if edge_df.empty:
        print("Graph rỗng.")
        return pd.DataFrame()

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    print(f"Community detection: {len(communities)} cộng đồng trên {G.number_of_nodes():,} node "
          f"/ {G.number_of_edges():,} cạnh (modularity greedy).")
    return pd.DataFrame(rows)

COMMUNITY_SYSTEM = """
You summarize a knowledge-graph community for a global-search index.
Base the summary strictly on the supplied relations. Return strict JSON only.
""".strip()

def summarize_communities(top_n=5, max_edges_per_community=40):
    """Sinh community report cho N cộng đồng lớn nhất -> tầng retrieval 'high-level'."""
    sizes = pd.DataFrame(run_cypher("""
    MATCH (n:Entity) WHERE n.community_id IS NOT NULL
    RETURN n.community_id AS community_id, count(n) AS size
    ORDER BY size DESC LIMIT $n
    """, n=int(top_n)))
    if sizes.empty:
        print("Chưa có community_id.")
        return pd.DataFrame()

    reports = []
    for r in tqdm(sizes.itertuples(index=False), total=len(sizes), desc="Community reports"):
        edges = run_cypher("""
        MATCH (a:Entity)-[rel]->(b:Entity)
        WHERE a.community_id=$cid AND b.community_id=$cid
        RETURN a.name AS a, type(rel) AS r, b.name AS b,
               rel.published_date AS d, rel.source_chunk_id AS c
        ORDER BY coalesce(rel.published_date,'') DESC
        LIMIT $lim
        """, cid=int(r.community_id), lim=int(max_edges_per_community))
        if not edges:
            continue
        lines = "\n".join(f"{e['a']} -{e['r']}-> {e['b']} | date={e['d']} | chunk={e['c']}"
                          for e in edges)
        try:
            obj, _ = groq_json(COMMUNITY_SYSTEM, f"""
RELATIONS:
{lines}

Return {{"title":"short name of this community","summary":"3-5 sentences","key_entities":["..."]}}
""")
        except Exception as e:
            obj = {"title": f"community_{r.community_id}", "summary": f"(lỗi: {e})", "key_entities": []}
        reports.append({
            "community_id": int(r.community_id), "size": int(r.size),
            "n_edges": len(edges),
            "title": norm_space(obj.get("title")),
            "summary": norm_space(obj.get("summary")),
            "key_entities": ", ".join(obj.get("key_entities", [])[:10]),
        })
    return pd.DataFrame(reports)

def answer_global(question, reports_df):
    """Global search: trả lời câu hỏi vĩ mô từ community reports thay vì từ chunk."""
    ctx = "\n\n".join(
        f"[community_id={r.community_id} | size={r.size}] {r.title}\n{r.summary}\nKey: {r.key_entities}"
        for r in reports_df.itertuples(index=False)
    )
    return generate_answer(question, f"=== COMMUNITY REPORTS ===\n{ctx}")

if RUN_BONUS:
    community_df = build_communities()
    _cr = OUT_DIR / "community_reports.csv"
    if _cr.exists() and not FORCE_RECOMPUTE:
        community_reports_df = pd.read_csv(_cr)
        print(f"resume: dung lai {len(community_reports_df)} community report da sinh.")
    else:
        community_reports_df = summarize_communities(top_n=5)
    display(community_reports_df)
    community_reports_df.to_csv(OUT_DIR / "community_reports.csv", index=False)

    _t0b, _s0b = time.perf_counter(), dict(LLM_STATS)
    GLOBAL_Q = "What are the dominant themes and the most active organizations across this news corpus?"
    try:
        _local = answer_flat_rag(GLOBAL_Q)
        _global = answer_global(GLOBAL_Q, community_reports_df)
    except Exception as _e:
        print(f"[bo qua] so sanh global — het quota LLM: {type(_e).__name__}")
        _local = _global = None
    if _local and _global:
        print("=== ĐỊNH LƯỢNG TRƯỚC/SAU (câu hỏi vĩ mô) ===")
        print(f"Flat RAG    : {_local['total_tokens']} tokens, {_local['e2e_latency_s']:.2f}s")
        print(f"Global(comm): {_global['total_tokens']} tokens, {_global['latency_s']:.2f}s")
        print("--- Flat RAG ---", _local["answer"][:600])
        print("--- Global Search ---", _global["answer"][:600])
    stage_report("Bonus A - Global Search", _t0b, _s0b)
else:
    print("RUN_BONUS = False -> bỏ qua.")

Community detection: 56 cộng đồng trên 286 node / 247 cạnh (modularity greedy).
resume: dung lai 5 community report da sinh.


,community_id,size,n_edges,title,summary,key_entities
0,0,36,35,IoT Industry Ecosystem and Leadership,"This community centers on the Internet of Things (IoT) sector, highlighting a broad coalition of technology companie...","IoT, IoT M2M Council, Romil Bahl, KORE, Consumer Technology Association, Vodafone, BICS, Telit, Aeris, Microsoft Azure"
1,1,30,36,Microsoft Ecosystem and Partnerships,"Microsoft is the central entity in this community, known for developing major software products such as Windows, Edg...","Microsoft, Windows, Edge, Copilot, Azure, KPMG, SAP, Nicole Herskowitz, Bill Chappell, Climate Innovation Fund"
2,2,19,27,Tech Giants and Innovation,"This community centers on major technology corporations, primarily Amazon and Meta, and their diverse development in...","Jeff Bezos, Amazon, Meta, Blue Origin, Facebook, Instagram, Google Meet, Climate Pledge Fund, Hugging Face, Pelocal ..."
3,3,17,16,Google Ecosystem and Partnerships,"This community centers on Google's diverse technological developments, including the Tensor Processing Unit, Google ...","Google, Samsung, Joey Lauffer, Eric Schmidt, Pixel Fold, Stadia, Tensor Processing Unit, Verizon, MediaTek, Galaxy W..."
4,4,14,40,Elon Musk's Tech Ventures,"Elon Musk leads and founded SpaceX, which developed the Starlink satellite internet service using low-orbit satellit...","Elon Musk, SpaceX, Starlink, X, Twitter"


   🛑 qwen/qwen3.6-27b cạn TPD (199702/200000 token/ngày). Chuyển sang model dự phòng.


=== ĐỊNH LƯỢNG TRƯỚC/SAU (câu hỏi vĩ mô) ===
Flat RAG    : 1427 tokens, 1.81s
Global(comm): 1413 tokens, 3.17s
--- Flat RAG --- **Dominant themes in the corpus**

| Theme | Evidence |
|-------|----------|
| **Artificial‑intelligence (AI) and generative‑AI** | “Top AI startup news… Anthropic hits the Google jackpot” (chunk 6c6c20da2a3046b0059b) and “AP Open AI agree to share select news content and technology” (chunk 67f61078101e32548302) highlight AI as a central topic. |
| **Tech startups & venture activity** | “National Beat: IBM’s startup shopping spree” (chunk 3f3995db06e48db870c3) and the Anthropic investment story show startup funding trends. |
| **Social‑media platform dynamics** | “Mark Zuckerberg… Threads hit 
--- Global Search --- **Dominant Themes**

| Theme | Key Indicators | Provenance |
|-------|----------------|------------|
| **Internet‑of‑Things (IoT) ecosystem & standards** | Focus on M2M Council, Consumer Technology Association, telecoms (Vodafone, BICS), IoT service

In [24]:
#@title Bonus B — Self-Correction Graph Retrieval (có định lượng trước/sau)
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

MAX_SELF_CORRECTION_ROUNDS = 2   # stop condition bắt buộc: hop2 -> hop3 -> vector fallback

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":"","rounds":0}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing,"rounds":1}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2,
        "rounds":MAX_SELF_CORRECTION_ROUNDS,
    }

def answer_self_correcting(question):
    t0 = time.perf_counter()
    sc = self_correcting_context(question)
    retrieval_s = time.perf_counter() - t0
    out = generate_answer(question, sc["context"])
    out.update({"context":sc["context"], "route":sc["route"], "rounds":sc["rounds"],
                "retrieval_latency_s":retrieval_s,
                "e2e_latency_s":retrieval_s + out["latency_s"]})
    return out

if RUN_BONUS and RUN_SELF_CORRECTION:
    # Chọn 2 câu mà GraphRAG chấm thấp nhất -> nơi self-correction có cơ hội cải thiện.
    _t0c, _s0c = time.perf_counter(), dict(LLM_STATS)
    _worst = eval_results_df.assign(
        _score=eval_results_df[["graph_comprehensiveness",
                                "graph_faithfulness",
                                "graph_multi_hop_reasoning"]].mean(axis=1)
    ).nsmallest(1, "_score")   # 1 cau du minh hoa; quota LLM co han

    sc_rows = []
    for q in _worst.itertuples(index=False):
        try:
            sc = answer_self_correcting(q.question)
            j = judge_answer(q.question, q.reference_answer, sc["answer"], sc["context"])
        except Exception as _e:
            print(f"[bo qua] self-correction {q.id} — het quota LLM: {type(_e).__name__}")
            continue
        before = np.mean([q.graph_comprehensiveness, q.graph_faithfulness,
                          q.graph_multi_hop_reasoning])
        after = np.mean([j["comprehensiveness"], j["faithfulness"], j["multi_hop_reasoning"]])
        sc_rows.append({
            "id": q.id, "group": q.group, "route": sc["route"], "rounds": sc["rounds"],
            "score_before": round(float(before), 3), "score_after": round(float(after), 3),
            "delta": round(float(after - before), 3),
            "latency_before_s": q.graph_latency_s,
            "latency_after_s": round(sc["e2e_latency_s"], 3),
            "tokens_before": q.graph_total_tokens, "tokens_after": sc["total_tokens"],
            "rationale_after": j["rationale"],
        })

    self_correction_df = pd.DataFrame(sc_rows)
    print("=== ĐỊNH LƯỢNG TRƯỚC/SAU SELF-CORRECTION ===")
    if self_correction_df.empty:
        print("(khong co ket qua — xem canh bao o tren)")
    else:
        display(self_correction_df)
        self_correction_df.to_csv(OUT_DIR / "self_correction_eval.csv", index=False)
    stage_report("Bonus B - Self-Correction", _t0c, _s0c)
else:
    print("Bo qua Bonus B (RUN_SELF_CORRECTION=False): quota LLM 200k token/ngay da can. "
          "Ham self_correcting_context() da dinh nghia va san sang chay khi quota hoi.")

Bo qua Bonus B (RUN_SELF_CORRECTION=False): quota LLM 200k token/ngay da can. Ham self_correcting_context() da dinh nghia va san sang chay khi quota hoi.


In [25]:
#@title 5.3 — Failure analysis helper + đóng gói nộp bài
# Cell này KHÔNG gọi LLM. Nó chỉ tổng hợp lại số liệu bạn cần để viết
# reports/failure_analysis.md và trả lời 10 câu thuyết minh.

_score_cols = ["comprehensiveness", "faithfulness", "multi_hop_reasoning"]
_fs = eval_results_df[[f"flat_{c}" for c in _score_cols]].mean(axis=1)
_gs = eval_results_df[[f"graph_{c}" for c in _score_cols]].mean(axis=1)
failure_df = eval_results_df.assign(
    flat_score=_fs.round(3), graph_score=_gs.round(3), delta=(_gs - _fs).round(3)
).sort_values("delta", ascending=False)

print("=" * 78)
print("CA 1 — GraphRAG THẮNG ĐẬM NHẤT (dùng cho 'Flat RAG thất bại / GraphRAG thành công')")
print("=" * 78)
_w = failure_df.iloc[0]
print(f"id={_w.id} | group={_w.group} | delta={_w.delta:+.2f} "
      f"(flat={_w.flat_score} vs graph={_w.graph_score})")
print(f"\nQ: {_w.question}\nREF: {_w.reference_answer}")
print(f"\n[Flat] seeds n/a | retrieved chunks: {_w.flat_retrieved_chunks}")
print(f"[Flat answer] {_w.flat_answer[:500]}")
print(f"[Flat judge]  {_w.flat_judge_rationale}")
print(f"\n[Graph] seeds={_w.graph_matched_seeds} | nodes={_w.graph_expanded_nodes} "
      f"| edges={_w.graph_collected_edges} | supernode_events={_w.graph_supernode_events}")
print(f"[Graph answer] {_w.graph_answer[:500]}")
print(f"[Graph judge]  {_w.graph_judge_rationale}")

print("\n" + "=" * 78)
print("CA 2 — GraphRAG THUA / KHÓ KHĂN NHẤT (dùng cho 'GraphRAG thất bại')")
print("=" * 78)
_l = failure_df.iloc[-1]
print(f"id={_l.id} | group={_l.group} | delta={_l.delta:+.2f} "
      f"(flat={_l.flat_score} vs graph={_l.graph_score})")
print(f"\nQ: {_l.question}\nREF: {_l.reference_answer}")
print(f"\n[Graph] seeds={_l.graph_matched_seeds} | nodes={_l.graph_expanded_nodes} "
      f"| edges={_l.graph_collected_edges} | supernode_events={_l.graph_supernode_events} "
      f"| global_cap_hit={_l.graph_global_cap_hit}")
print(f"[Graph answer] {_l.graph_answer[:500]}")
print(f"[Graph judge]  {_l.graph_judge_rationale}")
print("\n>>> Truy vết root-cause theo thứ tự: "
      "(1) seeds có khớp không? (2) cạnh cần thiết có trong triples_df không? "
      "(3) có bị supernode/global cap cắt không? (4) coref/extraction có sai không?")
display(failure_df[["id","group","flat_score","graph_score","delta",
                    "graph_matched_seeds","graph_collected_edges"]])
failure_df.to_csv(OUT_DIR / "failure_analysis_table.csv", index=False)

# ---------------- Checklist nộp bài ----------------
print("\n" + "=" * 78)
print("SUBMISSION CHECKLIST")
print("=" * 78)
_checks = [
    ("Neo4j connected + schema",        driver is not None),
    ("Exact dedup + chunking",          len(chunks_df) > 0),
    ("Near-dedup (bonus C)",            "near_dup_audit_df" in dir()),
    ("Coreference + unresolved log",    len(coref_df) > 0),
    ("Triples đúng schema allowlist",   len(triples_df) > 0),
    ("UNWIND bulk insert",              graph_counts["nodes"] > 0 and graph_counts["edges"] > 0),
    ("0 edge thiếu provenance",         graph_counts["invalid_provenance_edges"] == 0),
    ("Entity audit >= 10 dòng",         len(entity_resolution_audit_df) >= 10),
    ("Super-node policy đã chứng minh", bool(supernode_test_results)),
    ("Golden >= 5 câu, đủ 3 nhóm",      len(golden_df) >= 5 and golden_df.group.nunique() >= 3),
    ("reference_answer đã điền đủ",     not golden_df.reference_answer.fillna("").str.strip().eq("").any()),
    ("Eval chạy hết",                   len(eval_results_df) == len(golden_df)),
    ("Export eval_results.csv",         Path(EVAL_RESULTS_PATH).exists()),
    ("Export summary.csv",              Path(SUMMARY_PATH).exists()),
]
for _name, _ok in _checks:
    print(f"  [{'x' if _ok else ' '}] {_name}")

# ---------------- Đóng gói để tải về ----------------
_zip_base = str(BASE_DIR / "lab19_submission")
shutil.make_archive(_zip_base, "zip", root_dir=str(OUT_DIR))
print(f"\n📦 Đã nén outputs -> {_zip_base}.zip")
print(f"   Nội dung {OUT_DIR}:")
for _f in sorted(OUT_DIR.glob('*')):
    print(f"     - {_f.name} ({_f.stat().st_size:,} bytes)")

try:
    from google.colab import files
    print("\n👉 Chạy dòng dưới trong một cell mới để tải về máy:")
    print(f"   from google.colab import files; files.download('{_zip_base}.zip')")
    print(f"   files.download('{GOLDEN_PATH}')")
    print("   Và tải notebook: File > Download > Download .ipynb")
except Exception:
    pass

CA 1 — GraphRAG THẮNG ĐẬM NHẤT (dùng cho 'Flat RAG thất bại / GraphRAG thành công')
id=G05 | group=cross-doc | delta=+4.00 (flat=1.0 vs graph=5.0)

Q: Multiple articles mention both Elon Musk and SpaceX. Summarize what the corpus reports about their relationship across those articles, including the relationship types and publication dates.
REF: Across 13 distinct source chunks the corpus links Elon Musk to SpaceX via: FOUNDED, LEADS. Reported dates: 2023-01-12, 2023-07-27, 2023-10-07, 2023-07-07, 2023-01-09, 2023-02-22, 2023-02-03, 2023-01-14, 2023-01-13.

[Flat] seeds n/a | retrieved chunks: 136d721e010b5241dd0a::c0000; 6f87cb5e48c024983c7e::c0000; def538528cbd5aa93496::c0000; 7ec0e2da5a7dc0cf7f30::c0000; 5156990ad28770c257a4::c0000; 86cb5f9c878d3eb53479::c0000
[Flat answer] The corpus reports that Elon Musk is the CEO and leader of SpaceX [chunk_id=136d721e010b5241dd0a::c0000, chunk_id=6f87cb5e48c024983c7e::c0000, chunk_id=def538528cbd5aa93496::c0000]. The relationship is characteriz

,id,group,flat_score,graph_score,delta,graph_matched_seeds,graph_collected_edges
4,G05,cross-doc,1.000,5.0,4.000,Elon Musk[EXACT]; SpaceX[EXACT],47
3,G04,cross-doc,1.000,3.0,2.000,SpaceX[EXACT]; Starlink[EXACT],82
2,G03,multi-hop,4.333,5.0,0.667,Elon Musk[EXACT]; Starlink[EXACT],84
0,G01,factoid,5.000,5.0,0.000,audio video calling[EXACT],2
1,G02,factoid,5.000,5.0,0.000,X[EXACT],23



SUBMISSION CHECKLIST
  [x] Neo4j connected + schema
  [x] Exact dedup + chunking
  [x] Near-dedup (bonus C)
  [x] Coreference + unresolved log
  [x] Triples đúng schema allowlist
  [x] UNWIND bulk insert
  [x] 0 edge thiếu provenance
  [x] Entity audit >= 10 dòng
  [x] Super-node policy đã chứng minh
  [x] Golden >= 5 câu, đủ 3 nhóm
  [x] reference_answer đã điền đủ
  [x] Eval chạy hết
  [x] Export eval_results.csv
  [x] Export summary.csv



📦 Đã nén outputs -> d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\lab19_submission.zip


   Nội dung d:\VinUni_AI20K\Lab\DAY19_2A202601585_ThachMinhQuan\outputs:


     - .gitkeep (76 bytes)
     - _stale (0 bytes)
     - community_reports.csv (3,422 bytes)
     - coref_checkpoint.csv (87,210 bytes)
     - coref_diff_report.csv (62,406 bytes)
     - entity_resolution_audit.csv (1,125 bytes)
     - extraction_checkpoint.csv (46,836 bytes)
     - extraction_checkpoint.raw.jsonl (86,479 bytes)
     - failure_analysis_table.csv (60,711 bytes)
     - graphrag_eval_checkpoint.csv (60,618 bytes)
     - graphrag_eval_results.csv (60,618 bytes)
     - graphrag_vs_flatrag_summary.csv (2,310 bytes)
     - near_dedup_audit.csv (1,067 bytes)
     - near_dedup_threshold_sweep.csv (94 bytes)
     - supernode_report.csv (695 bytes)


# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau